# Statistical Feature Research

This notebook develops an independent statistical research branch for GC/MGC. It validates the trusted active-contract table, constructs the GC eligible-observation frame, builds leakage-free fixed-horizon forward labels, and establishes the unconditional GC baseline benchmark. It does not load raw DBN data or POI outputs, introduce R-multiple outcomes, engineer predictors, simulate trades, or begin Section 6.


# 1.0 Configuration and Research Environment

This section establishes a reproducible project environment and locks the market-time conventions that later research must obey. All paths are resolved relative to the project root, UTC timestamps remain reference keys, and New York time is authoritative for every trading-session rule.

## 1.1 Imports

Only standard numerical, tabular, timezone, and Parquet dependencies are imported at this stage. Package versions and the running executable are printed so that the executed notebook records its research environment.

In [1]:
from __future__ import annotations

import random
import re
import sys
from datetime import time
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

print(f"Python {sys.version.split()[0]}")
print(f"Executable: {sys.executable}")
print(f"pandas {pd.__version__} | numpy {np.__version__} | pyarrow {pa.__version__}")

Python 3.14.5
Executable: C:\Users\abond\Desktop\WORK FILES\Systemic\Project 1\.venv-1\Scripts\python.exe
pandas 3.0.3 | numpy 2.4.6 | pyarrow 24.0.0


## 1.2 Paths

The project root is discovered by walking upward from the current working directory until both trusted project markers are found. The notebook therefore works whether launched from the project root or a notebook subdirectory. The active interpreter is required to be the project's `.venv-1` environment.

In [2]:
TRUSTED_DATA_RELATIVE_PATH = Path("data/processed/research_bars_gc_mgc_1m.parquet")
CONTEXT_RELATIVE_PATH = Path("project_docs/statistical_feature_research_context_report.md")
NOTEBOOK_RELATIVE_PATH = Path("notebooks/exploration/statistical_feature_research.ipynb")


def find_project_root(start: Path | None = None) -> Path:
    """Locate the project without relying on the notebook's launch directory."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / TRUSTED_DATA_RELATIVE_PATH).is_file() and (candidate / CONTEXT_RELATIVE_PATH).is_file():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / TRUSTED_DATA_RELATIVE_PATH
CONTEXT_REPORT_PATH = PROJECT_ROOT / CONTEXT_RELATIVE_PATH
NOTEBOOK_PATH = PROJECT_ROOT / NOTEBOOK_RELATIVE_PATH
EXPECTED_PYTHON = PROJECT_ROOT / ".venv-1" / "Scripts" / "python.exe"

assert DATA_PATH.is_file(), f"Missing trusted dataset: {DATA_PATH}"
assert CONTEXT_REPORT_PATH.is_file(), f"Missing governing report: {CONTEXT_REPORT_PATH}"
assert DATA_PATH.name == "research_bars_gc_mgc_1m.parquet" and DATA_PATH.suffix == ".parquet"
assert not any(token in DATA_PATH.name.lower() for token in (".dbn", "section6_", "section7_", "poi"))
assert Path(sys.executable).resolve() == EXPECTED_PYTHON.resolve(), (
    f"Wrong interpreter. Expected {EXPECTED_PYTHON}, received {sys.executable}."
)

environment_check = pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "trusted_data": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "governing_context": str(CONTEXT_REPORT_PATH.relative_to(PROJECT_ROOT)),
        "active_interpreter": str(Path(sys.executable).resolve()),
        "expected_environment": ".venv-1",
        "environment_valid": True,
    },
    name="value",
)
display(environment_check.to_frame())

,value
project_root,C:\Users\abond\Desktop\WORK FILES\Systemic\Pro...
trusted_data,data\processed\research_bars_gc_mgc_1m.parquet
governing_context,project_docs\statistical_feature_research_cont...
active_interpreter,C:\Users\abond\Desktop\WORK FILES\Systemic\Pro...
expected_environment,.venv-1
environment_valid,True


## 1.3 Instrument and Time Settings

GC and MGC are the only expected products, with GC designated for the later Phase 1 discovery population. `America/New_York` is the authoritative timezone for session classification, trading dates, entry eligibility, the noon cutoff, and the 3:30pm mandatory exit. UTC is retained only as an immutable timestamp reference. The entry intervals are recorded now but are not applied until the eligible observation frame is constructed in a later section.

In [3]:
EXPECTED_PRODUCTS = ("GC", "MGC")
PRIMARY_RESEARCH_PRODUCT = "GC"
RESEARCH_TIMEZONE_NAME = "America/New_York"
REFERENCE_TIMEZONE_NAME = "UTC"
RESEARCH_TIMEZONE = ZoneInfo(RESEARCH_TIMEZONE_NAME)
REFERENCE_TIMEZONE = ZoneInfo(REFERENCE_TIMEZONE_NAME)
UTC_REFERENCE_KEY_COLUMNS = ("ts_event_utc", "product")

ENTRY_WINDOWS_NY = {
    "London": (time(3, 0), time(6, 0)),
    "New York": (time(7, 0), time(12, 0)),
}
NO_NEW_ENTRY_AT_OR_AFTER_NY = time(12, 0)
MANDATORY_EXIT_NY = time(15, 30)

assert PRIMARY_RESEARCH_PRODUCT in EXPECTED_PRODUCTS
assert RESEARCH_TIMEZONE.key == "America/New_York"
assert ENTRY_WINDOWS_NY["London"] == (time(3, 0), time(6, 0))
assert ENTRY_WINDOWS_NY["New York"] == (time(7, 0), time(12, 0))
assert NO_NEW_ENTRY_AT_OR_AFTER_NY == time(12, 0)
assert MANDATORY_EXIT_NY == time(15, 30)

time_settings = pd.Series(
    {
        "expected_products": EXPECTED_PRODUCTS,
        "phase_1_product": PRIMARY_RESEARCH_PRODUCT,
        "research_timezone": RESEARCH_TIMEZONE_NAME,
        "UTC_reference_keys": UTC_REFERENCE_KEY_COLUMNS,
        "London_entry_window_NY": "[03:00, 06:00)",
        "New_York_entry_window_NY": "[07:00, 12:00)",
        "mandatory_exit_NY": "15:30",
    },
    name="setting",
)
display(time_settings.to_frame())

,setting
expected_products,"(GC, MGC)"
phase_1_product,GC
research_timezone,America/New_York
UTC_reference_keys,"(ts_event_utc, product)"
London_entry_window_NY,"[03:00, 06:00)"
New_York_entry_window_NY,"[07:00, 12:00)"
mandatory_exit_NY,15:30


## 1.4 Random Seeds and Display Settings

A fixed seed is set for reproducible sampling and later statistical procedures. Display settings expose enough rows and columns to audit the saved schema and validation tables without changing analytical results.

In [4]:
RANDOM_SEED = 20260714
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

print(f"Random seed fixed at {RANDOM_SEED}.")

Random seed fixed at 20260714.


# 2.0 Load Trusted Research Data

The sole market-data input is the completed active-contract research table. The saved Arrow schema is inspected before the table is materialized; no raw DBN archive, upstream active-contract logic, or Section 6/7 POI output is read.

## 2.1 Load Final GC/MGC Research Table

Parquet metadata and the physical Arrow schema are read first. The complete table is then loaded once with Arrow-backed pandas dtypes to reduce memory overhead while preserving the saved timezone and integer types. Subsequent checks reuse this single dataframe.

In [5]:
parquet_file = pq.ParquetFile(DATA_PATH)
saved_arrow_schema = parquet_file.schema_arrow
saved_column_names = tuple(saved_arrow_schema.names)
parquet_metadata = pd.Series(
    {
        "relative_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "rows": parquet_file.metadata.num_rows,
        "columns": parquet_file.metadata.num_columns,
        "row_groups": parquet_file.metadata.num_row_groups,
        "file_size_MiB": DATA_PATH.stat().st_size / (1024**2),
    },
    name="saved_value",
)
display(parquet_metadata.to_frame())

research_bars = pd.read_parquet(DATA_PATH, engine="pyarrow", dtype_backend="pyarrow")

assert len(research_bars) == parquet_file.metadata.num_rows
assert tuple(research_bars.columns) == saved_column_names
loaded_summary = pd.Series(
    {
        "loaded_rows": len(research_bars),
        "loaded_columns": research_bars.shape[1],
        "dataframe_memory_MiB": research_bars.memory_usage(index=True, deep=True).sum() / (1024**2),
        "dtype_backend": "pyarrow",
    },
    name="loaded_value",
)
display(loaded_summary.to_frame())

,saved_value
relative_path,data\processed\research_bars_gc_mgc_1m.parquet
rows,3487656
columns,82
row_groups,4
file_size_MiB,603.389116


,loaded_value
loaded_rows,3487656
loaded_columns,82
dataframe_memory_MiB,"1,890.141650"
dtype_backend,pyarrow


## 2.2 Confirm Schema and Data Types

The saved Arrow type and loaded pandas type are shown for every column before any downstream assumptions are made. Required identifiers, market fields, session fields, and controls are asserted. Product counts, within-product timestamp order, UTC-key uniqueness, and core OHLCV completeness are validated.

Existing columns whose names indicate a future market outcome are placed in `EXISTING_FORWARD_COLUMNS`. This quarantine is a namespace safeguard: these columns remain available only for possible later validation and must never enter the independent feature frame. Operational schedule fields such as `days_to_next_roll` are rollover controls, not realized market outcomes, and are not classified as feature labels here. Timestamp sorting is required within each product; no assumption is made about the panel's physical interleaving across products.

In [6]:
schema_table = pd.DataFrame(
    {
        "column": saved_column_names,
        "saved_arrow_type": [str(field.type) for field in saved_arrow_schema],
        "loaded_pandas_type": [str(research_bars.dtypes[name]) for name in saved_column_names],
    }
)
display(schema_table)

REQUIRED_COLUMNS = {
    "ts_event_utc", "ts_event_ny", "trade_date_utc", "trade_date_ny",
    "product", "symbol", "active_symbol", "open", "high", "low", "close", "volume",
    "hour_ny", "minute_ny", "session_label", "continuous_segment_id",
    "is_roll_day", "is_pre_roll_day", "is_post_roll_day", "roll_window_flag",
    "roll_window_type", "days_since_roll", "days_to_next_roll", "tradable_research_flag",
}
CORE_OHLCV_COLUMNS = ["open", "high", "low", "close", "volume"]
EXPECTED_ROW_COUNTS = {"GC": 1_759_671, "MGC": 1_727_985}
OUTCOME_NAME_PATTERNS = (
    r"(^|_)forward(_|$)",
    r"(^|_)future(_|$)",
    r"(^|_)(mfe|mae)(_|$)",
    r"(^|_)(target|stop).*(hit|outcome)",
    r"(^|_)outcome(_|$)",
)
EXISTING_FORWARD_COLUMNS = [
    name
    for name in saved_column_names
    if any(re.search(pattern, name.lower()) for pattern in OUTCOME_NAME_PATTERNS)
]

missing_required_columns = sorted(REQUIRED_COLUMNS.difference(research_bars.columns))
product_counts = research_bars["product"].value_counts(sort=False).sort_index()
per_product_timestamp_sorted = {
    product: research_bars.loc[research_bars["product"].eq(product), "ts_event_utc"].is_monotonic_increasing
    for product in EXPECTED_PRODUCTS
}
global_timestamp_sorted = research_bars["ts_event_utc"].is_monotonic_increasing
duplicate_timestamp_product_rows = int(research_bars.duplicated(list(UTC_REFERENCE_KEY_COLUMNS)).sum())
core_ohlcv_missing = research_bars[CORE_OHLCV_COLUMNS].isna().sum()

schema_checks = pd.Series(
    {
        "all_required_columns_present": not missing_required_columns,
        "expected_products_only": set(product_counts.index) == set(EXPECTED_PRODUCTS),
        "expected_product_row_counts": product_counts.to_dict() == EXPECTED_ROW_COUNTS,
        "timestamps_sorted_within_each_product": all(per_product_timestamp_sorted.values()),
        "duplicate_timestamp_product_rows_zero": duplicate_timestamp_product_rows == 0,
        "core_OHLCV_missing_values_zero": int(core_ohlcv_missing.sum()) == 0,
        "existing_forward_columns_quarantined": len(EXISTING_FORWARD_COLUMNS) > 0,
    },
    name="passed",
)

display(product_counts.rename("rows").to_frame())
display(pd.DataFrame({"timestamp_sorted": per_product_timestamp_sorted}))
display(core_ohlcv_missing.rename("missing_values").to_frame())
print(f"Global physical timestamp order (informational): {global_timestamp_sorted}")
print(f"Duplicate (ts_event_utc, product) rows: {duplicate_timestamp_product_rows:,}")
print(f"EXISTING_FORWARD_COLUMNS ({len(EXISTING_FORWARD_COLUMNS)}): {EXISTING_FORWARD_COLUMNS}")
display(schema_checks.to_frame())

assert schema_checks.all(), schema_checks[~schema_checks].index.tolist()

,column,saved_arrow_type,loaded_pandas_type
0,ts_event_utc,"timestamp[ns, tz=UTC]","timestamp[ns, tz=UTC][pyarrow]"
1,product,large_string,large_string[pyarrow]
2,symbol,large_string,large_string[pyarrow]
3,rtype,uint8,uint8[pyarrow]
4,publisher_id,uint16,uint16[pyarrow]
5,instrument_id,uint32,uint32[pyarrow]
6,open,double,double[pyarrow]
7,high,double,double[pyarrow]
8,low,double,double[pyarrow]
9,close,double,double[pyarrow]


,rows
product,
GC,1759671
MGC,1727985


,timestamp_sorted
GC,True
MGC,True


,missing_values
open,0
high,0
low,0
close,0
volume,0


Global physical timestamp order (informational): False
Duplicate (ts_event_utc, product) rows: 0
EXISTING_FORWARD_COLUMNS (8): ['forward_return_5m', 'forward_log_return_5m', 'forward_return_15m', 'forward_log_return_15m', 'forward_return_30m', 'forward_log_return_30m', 'forward_return_60m', 'forward_log_return_60m']


,passed
all_required_columns_present,True
expected_products_only,True
expected_product_row_counts,True
timestamps_sorted_within_each_product,True
duplicate_timestamp_product_rows_zero,True
core_OHLCV_missing_values_zero,True
existing_forward_columns_quarantined,True


## 2.3 Confirm Timestamp and Time-Zone Handling

The physical timezone metadata is validated directly from the saved schema. Each New York timestamp must represent the same instant as its UTC reference, minute alignment must be exact, the saved New York calendar date and clock fields must agree with `ts_event_ny`, and UTC timestamps are not reinterpreted as session clock time.

In [7]:
utc_arrow_type = saved_arrow_schema.field("ts_event_utc").type
ny_arrow_type = saved_arrow_schema.field("ts_event_ny").type

timezone_checks = pd.Series(
    {
        "ts_event_utc_is_timezone_aware_UTC": pa.types.is_timestamp(utc_arrow_type) and utc_arrow_type.tz == "UTC",
        "ts_event_ny_is_timezone_aware_New_York": pa.types.is_timestamp(ny_arrow_type) and ny_arrow_type.tz == RESEARCH_TIMEZONE_NAME,
        "UTC_and_New_York_columns_represent_same_instants": bool(
            research_bars["ts_event_ny"].dt.tz_convert(REFERENCE_TIMEZONE_NAME).eq(research_bars["ts_event_utc"]).all()
        ),
        "UTC_timestamps_are_exactly_minute_aligned": bool(
            research_bars["ts_event_utc"].eq(research_bars["ts_event_utc"].dt.floor("min")).all()
        ),
        "New_York_trade_dates_match_local_calendar_dates": bool(
            research_bars["ts_event_ny"].dt.date.eq(research_bars["trade_date_ny"].dt.date).all()
        ),
        "saved_hour_ny_matches_local_timestamp": bool(
            research_bars["ts_event_ny"].dt.hour.eq(research_bars["hour_ny"]).all()
        ),
        "saved_minute_ny_matches_local_timestamp": bool(
            research_bars["ts_event_ny"].dt.minute.eq(research_bars["minute_ny"]).all()
        ),
    },
    name="passed",
)
display(timezone_checks.to_frame())
display(research_bars[["ts_event_utc", "ts_event_ny", "trade_date_ny", "hour_ny", "minute_ny"]].head(3))

assert timezone_checks.all(), timezone_checks[~timezone_checks].index.tolist()

,passed
ts_event_utc_is_timezone_aware_UTC,True
ts_event_ny_is_timezone_aware_New_York,True
UTC_and_New_York_columns_represent_same_instants,True
UTC_timestamps_are_exactly_minute_aligned,True
New_York_trade_dates_match_local_calendar_dates,True
saved_hour_ny_matches_local_timestamp,True
saved_minute_ny_matches_local_timestamp,True


,ts_event_utc,ts_event_ny,trade_date_ny,hour_ny,minute_ny
0,2021-05-24 00:00:00+00:00,2021-05-23 20:00:00-04:00,2021-05-23 00:00:00,20,0
1,2021-05-24 00:01:00+00:00,2021-05-23 20:01:00-04:00,2021-05-23 00:00:00,20,1
2,2021-05-24 00:02:00+00:00,2021-05-23 20:02:00-04:00,2021-05-23 00:00:00,20,2


## 2.4 Confirm Date Coverage

Coverage is summarized separately for GC and MGC using both UTC event timestamps and New York trading dates. The two product histories are required to share the reported endpoints and trading-date count. Ordinary futures closures are not treated as missing data in this coverage check.

In [8]:
coverage_by_product = (
    research_bars.groupby("product", observed=True)
    .agg(
        rows=("product", "size"),
        first_ts_utc=("ts_event_utc", "min"),
        last_ts_utc=("ts_event_utc", "max"),
        first_trade_date_ny=("trade_date_ny", "min"),
        last_trade_date_ny=("trade_date_ny", "max"),
        trading_dates=("trade_date_ny", "nunique"),
    )
    .sort_index()
)
display(coverage_by_product)

expected_coverage = {
    "first_ts_utc": pd.Timestamp("2021-05-24 00:00:00", tz="UTC"),
    "last_ts_utc": pd.Timestamp("2026-05-22 20:59:00", tz="UTC"),
    "first_trade_date_ny": pd.Timestamp("2021-05-23"),
    "last_trade_date_ny": pd.Timestamp("2026-05-22"),
    "trading_dates": 1_556,
}
coverage_checks = pd.Series(
    {
        name: bool(coverage_by_product[name].eq(expected).all())
        for name, expected in expected_coverage.items()
    },
    name="passed",
)
display(coverage_checks.to_frame())

assert coverage_checks.all(), coverage_checks[~coverage_checks].index.tolist()

,rows,first_ts_utc,last_ts_utc,first_trade_date_ny,last_trade_date_ny,trading_dates
product,,,,,,
GC,1759671,2021-05-24 00:00:00+00:00,2026-05-22 20:59:00+00:00,2021-05-23 00:00:00,2026-05-22 00:00:00,1556
MGC,1727985,2021-05-24 00:00:00+00:00,2026-05-22 20:59:00+00:00,2021-05-23 00:00:00,2026-05-22 00:00:00,1556


,passed
first_ts_utc,True
last_ts_utc,True
first_trade_date_ny,True
last_trade_date_ny,True
trading_dates,True


## 2.5 Confirm Rollover and Exclusion Flags

Rollover, boundary, and research-tradability controls are validated as saved inputs rather than reconstructed. Boolean values are summarized by product, rollover categories and transition labels are enumerated, and day-level roll-distance fields are described. Missing roll distances are retained at leading/trailing coverage boundaries, while `roll_transition_label` is expected to be populated only on roll days. No eligibility filter is applied in this section.

In [9]:
ROLLOVER_AND_EXCLUSION_COLUMNS = [
    "is_roll_day",
    "is_pre_roll_day",
    "is_post_roll_day",
    "roll_window_flag",
    "roll_window_type",
    "roll_group_id",
    "days_since_roll",
    "days_to_next_roll",
    "roll_transition_label",
    "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag",
    "tradable_research_flag",
]
BOOLEAN_CONTROL_COLUMNS = [
    "is_roll_day",
    "is_pre_roll_day",
    "is_post_roll_day",
    "roll_window_flag",
    "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag",
    "tradable_research_flag",
]
missing_control_columns = sorted(set(ROLLOVER_AND_EXCLUSION_COLUMNS).difference(research_bars.columns))
control_missing_values = research_bars[ROLLOVER_AND_EXCLUSION_COLUMNS].isna().sum()

boolean_true_counts = (
    research_bars.groupby("product", observed=True)[BOOLEAN_CONTROL_COLUMNS].sum().astype("int64")
)
boolean_total_counts = research_bars.groupby("product", observed=True).size().astype("int64")
boolean_summary_records = []
for product in EXPECTED_PRODUCTS:
    for column in BOOLEAN_CONTROL_COLUMNS:
        true_count = int(boolean_true_counts.loc[product, column])
        total_count = int(boolean_total_counts.loc[product])
        boolean_summary_records.append(
            {
                "product": product,
                "field": column,
                "false_rows": total_count - true_count,
                "true_rows": true_count,
                "true_rate": true_count / total_count,
            }
        )
boolean_control_summary = pd.DataFrame(boolean_summary_records).set_index(["product", "field"])

roll_window_type_summary = (
    research_bars.groupby(["product", "roll_window_type"], observed=True, dropna=False)
    .size()
    .rename("rows")
    .to_frame()
)
roll_transition_summary = (
    research_bars.groupby(["product", "roll_transition_label"], observed=True, dropna=False)
    .size()
    .rename("rows")
    .to_frame()
)
daily_roll_state = research_bars[
    ["product", "trade_date_ny", "roll_group_id", "days_since_roll", "days_to_next_roll"]
].drop_duplicates()
roll_distance_summary = daily_roll_state.groupby("product", observed=True)[
    ["roll_group_id", "days_since_roll", "days_to_next_roll"]
].agg(["min", "median", "max"])

display(boolean_control_summary)
display(roll_window_type_summary)
display(roll_transition_summary)
display(roll_distance_summary)
display(control_missing_values.rename("missing_values").to_frame())

control_checks = pd.Series(
    {
        "all_rollover_and_exclusion_fields_present": not missing_control_columns,
        "required_boolean_and_categorical_controls_nonmissing": int(
            control_missing_values[BOOLEAN_CONTROL_COLUMNS + ["roll_window_type", "roll_group_id"]].sum()
        ) == 0,
        "roll_transition_label_present_exactly_on_roll_days": bool(
            research_bars["roll_transition_label"].notna().eq(research_bars["is_roll_day"]).all()
        ),
        "roll_distance_boundary_missingness_is_explicit": bool(
            control_missing_values[["days_since_roll", "days_to_next_roll"]].gt(0).all()
        ),
        "roll_window_flag_matches_roll_window_type": bool(
            research_bars["roll_window_flag"].eq(research_bars["roll_window_type"].ne("none")).all()
        ),
        "tradable_research_flag_has_true_and_false_rows": set(
            research_bars["tradable_research_flag"].unique().tolist()
        ) == {False, True},
    },
    name="passed",
)
display(control_checks.to_frame())

assert control_checks.all(), control_checks[~control_checks].index.tolist()

false_rows  true_rows  true_rate
product field                                                          
GC      is_roll_day                       1727634      32037   0.018206
        is_pre_roll_day                   1738275      21396   0.012159
        is_post_roll_day                  1735453      24218   0.013763
        roll_window_flag                  1682020      77651   0.044128
        low_liquidity_active_day_flag     1739487      20184   0.011470
        low_liquidity_warning_flag        1739487      20184   0.011470
        tradable_research_flag              83080    1676591   0.952787
MGC     is_roll_day                       1697290      30695   0.017763
        is_pre_roll_day                   1716268      11717   0.006781
        is_post_roll_day                  1701656      26329   0.015237
        roll_window_flag                  1659244      68741   0.039781
        low_liquidity_active_day_flag     1706300      21685   0.012549
        low_liquidity_warning_flag        1706300      21685   0.012549
        tradable_research_flag              78709    1649276   0.954450

rows
product roll_window_type         
GC      none              1682020
        post_roll_day       24218
        pre_roll_day        21396
        roll_day            32037
MGC     none              1659244
        post_roll_day       26329
        pre_roll_day        11717
        roll_day            30695

rows
product roll_transition_label         
GC      GCG2 -> GCJ2              1376
        GCG3 -> GCJ3              1380
        GCG4 -> GCJ4              1376
        GCG5 -> GCJ5              1319
        GCG6 -> GCJ6              1380
        GCJ2 -> GCM2              1371
        GCJ3 -> GCM3              1379
        GCJ4 -> GCM4              1260
        GCJ5 -> GCM5               120
        GCJ6 -> GCM6              1380
        GCM1 -> GCQ1              1260
        GCM2 -> GCQ2              1228
        GCM3 -> GCQ3              1371
        GCM4 -> GCQ4              1260
        GCM5 -> GCQ5              1380
        GCQ1 -> GCZ1              1260
        GCQ2 -> GCZ2              1260
        GCQ3 -> GCZ3              1375
        GCQ4 -> GCZ4              1379
        GCQ5 -> GCZ5              1379
        GCZ1 -> GCG2              1379
        GCZ2 -> GCG3              1379
        GCZ3 -> GCG4              1380
        GCZ4 -> GCG5              1184
        GCZ5 -> GCG6              1222
        <NA>                   1727634
MGC     MGCG2 -> MGCJ2            1317
        MGCG3 -> MGCJ3            1369
        MGCG4 -> MGCJ4            1361
        MGCG5 -> MGCJ5            1320
        MGCG6 -> MGCJ6            1380
        MGCJ2 -> MGCM2            1371
        MGCJ3 -> MGCM3             120
        MGCJ4 -> MGCM4            1256
        MGCJ5 -> MGCM5            1380
        MGCJ6 -> MGCM6            1380
        MGCM1 -> MGCQ1             100
        MGCM2 -> MGCQ2            1179
        MGCM3 -> MGCQ3            1368
        MGCM4 -> MGCQ4            1258
        MGCM5 -> MGCQ5            1260
        MGCQ1 -> MGCZ1            1325
        MGCQ2 -> MGCZ2            1317
        MGCQ3 -> MGCZ3            1365
        MGCQ4 -> MGCZ4            1380
        MGCQ5 -> MGCZ5            1380
        MGCZ1 -> MGCG2            1364
        MGCZ2 -> MGCG3            1362
        MGCZ3 -> MGCG4            1370
        MGCZ4 -> MGCG5            1185
        MGCZ5 -> MGCG6            1228
        <NA>                   1697290

roll_group_id               days_since_roll                days_to_next_roll               
                  min    median max             min    median  max               min    median  max
product                                                                                            
GC                  1 13.000000  26               0 30.500000  104                 0 31.000000  104
MGC                 1 13.000000  26               0 30.500000  103                 0 31.000000  103

,missing_values
is_roll_day,0
is_pre_roll_day,0
is_post_roll_day,0
roll_window_flag,0
roll_window_type,0
roll_group_id,0
days_since_roll,9982
days_to_next_roll,101823
roll_transition_label,3424924
low_liquidity_active_day_flag,0


,passed
all_rollover_and_exclusion_fields_present,True
required_boolean_and_categorical_controls_nonmissing,True
roll_transition_label_present_exactly_on_roll_days,True
roll_distance_boundary_missingness_is_explicit,True
roll_window_flag_matches_roll_window_type,True
tradable_research_flag_has_true_and_false_rows,True


### Section 2 validation summary

The executable summary below combines the schema, product, timestamp, timezone, coverage, missingness, rollover/tradability, and forward-column quarantine gates. `READY` means the trusted dataset is suitable for the next milestone—construction of the eligible observation frame—but that construction is intentionally outside the scope of this notebook increment.

In [10]:
section_2_validation = pd.concat(
    [schema_checks, timezone_checks, coverage_checks, control_checks],
    keys=["schema_and_content", "timestamps_and_timezones", "coverage", "rollover_and_exclusions"],
).rename("passed")
ready_for_eligible_observation_frame = bool(section_2_validation.all())

display(section_2_validation.to_frame())
print(
    "Section 2 status: "
    + (
        "READY — the trusted GC/MGC table is validated for construction of the eligible observation frame."
        if ready_for_eligible_observation_frame
        else "NOT READY — at least one validation gate failed."
    )
)
print(f"Quarantined outcome-derived columns: {EXISTING_FORWARD_COLUMNS}")

assert ready_for_eligible_observation_frame

passed
schema_and_content       all_required_columns_present                          True
                         expected_products_only                                True
                         expected_product_row_counts                           True
                         timestamps_sorted_within_each_product                 True
                         duplicate_timestamp_product_rows_zero                 True
                         core_OHLCV_missing_values_zero                        True
                         existing_forward_columns_quarantined                  True
timestamps_and_timezones ts_event_utc_is_timezone_aware_UTC                    True
                         ts_event_ny_is_timezone_aware_New_York                True
                         UTC_and_New_York_columns_represent_same_instants      True
                         UTC_timestamps_are_exactly_minute_aligned             True
                         New_York_trade_dates_match_local_calendar_dates       True
                         saved_hour_ny_matches_local_timestamp                 True
                         saved_minute_ny_matches_local_timestamp               True
coverage                 first_ts_utc                                          True
                         last_ts_utc                                           True
                         first_trade_date_ny                                   True
                         last_trade_date_ny                                    True
                         trading_dates                                         True
rollover_and_exclusions  all_rollover_and_exclusion_fields_present             True
                         required_boolean_and_categorical_controls_nonmi...    True
                         roll_transition_label_present_exactly_on_roll_days    True
                         roll_distance_boundary_missingness_is_explicit        True
                         roll_window_flag_matches_roll_window_type             True
                         tradable_research_flag_has_true_and_false_rows        True

Section 2 status: READY — the trusted GC/MGC table is validated for construction of the eligible observation frame.
Quarantined outcome-derived columns: ['forward_return_5m', 'forward_log_return_5m', 'forward_return_15m', 'forward_log_return_15m', 'forward_return_30m', 'forward_log_return_30m', 'forward_return_60m', 'forward_log_return_60m']


# 3.0 Construct the Eligible Observation Frame

This section maps each valid completed GC decision bar `t` to the true next one-minute bar `t+1`, whose open will be the theoretical entry in later research. Eligibility is determined from the entry timestamp in `America/New_York` and from both bars' saved contract, segment, rollover, liquidity, and tradability state. No prices, forward outcomes, labels, or statistical features are constructed here.

## 3.1 GC Phase 1 Research Population

Phase 1 uses GC only, while the trusted `research_bars` dataframe remains unchanged and continues to contain MGC. A narrow Arrow-backed working table retains only identifiers, timestamps, contract/segment identity, and the saved rollover, liquidity, and tradability controls needed to construct and audit observation pairs. The actual schema provides `symbol`, `active_symbol`, `instrument_id`, and `continuous_segment_id`, so no boundary fallback is required.

In [11]:
assert ready_for_eligible_observation_frame
assert isinstance(research_bars.index, pd.RangeIndex), "Stable source-row identifiers require the trusted RangeIndex."

SECTION_3_SOURCE_COLUMNS = [
    "ts_event_utc",
    "ts_event_ny",
    "trade_date_ny",
    "product",
    "symbol",
    "active_symbol",
    "instrument_id",
    "continuous_segment_id",
    "tradable_research_flag",
    "is_roll_day",
    "is_pre_roll_day",
    "is_post_roll_day",
    "roll_window_flag",
    "roll_window_type",
    "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag",
]
missing_section_3_columns = sorted(set(SECTION_3_SOURCE_COLUMNS).difference(research_bars.columns))
assert not missing_section_3_columns, missing_section_3_columns

gc_source_mask = research_bars["product"].eq(PRIMARY_RESEARCH_PRODUCT)
gc_source_row_ids = np.flatnonzero(gc_source_mask.to_numpy(dtype=bool))
gc_section_3 = research_bars.loc[gc_source_mask, SECTION_3_SOURCE_COLUMNS].copy()
gc_section_3.insert(0, "source_row_id", pd.array(gc_source_row_ids, dtype="int64[pyarrow]"))
gc_source_rows = len(gc_section_3)

derived_tradable_flag = ~(
    gc_section_3["roll_window_flag"]
    | gc_section_3["low_liquidity_active_day_flag"]
    | gc_section_3["low_liquidity_warning_flag"]
)
gc_population_checks = pd.Series(
    {
        "expected_GC_source_rows": gc_source_rows == EXPECTED_ROW_COUNTS[PRIMARY_RESEARCH_PRODUCT],
        "GC_only": set(gc_section_3["product"].unique().tolist()) == {PRIMARY_RESEARCH_PRODUCT},
        "timestamps_sorted": gc_section_3["ts_event_utc"].is_monotonic_increasing,
        "source_row_ids_unique": gc_section_3["source_row_id"].is_unique,
        "selected_symbol_matches_active_symbol": bool(
            gc_section_3["symbol"].eq(gc_section_3["active_symbol"]).all()
        ),
        "outright_GC_symbol_format_only": bool(
            gc_section_3["symbol"].str.fullmatch(r"GC[A-Z]\d{1,2}").all()
        ),
        "contract_segment_identifiers_complete": bool(
            gc_section_3[["symbol", "active_symbol", "instrument_id", "continuous_segment_id"]]
            .notna()
            .all(axis=None)
        ),
        "tradable_flag_matches_saved_controls": bool(
            gc_section_3["tradable_research_flag"].eq(derived_tradable_flag).all()
        ),
        "trusted_source_still_contains_MGC": "MGC" in set(research_bars["product"].unique().tolist()),
    },
    name="passed",
)
display(pd.Series({"GC_source_rows": gc_source_rows, "working_columns": gc_section_3.shape[1]}, name="value").to_frame())
display(gc_population_checks.to_frame())
assert gc_population_checks.all(), gc_population_checks[~gc_population_checks].index.tolist()

,value
GC_source_rows,1759671
working_columns,17


,passed
expected_GC_source_rows,True
GC_only,True
timestamps_sorted,True
source_row_ids_unique,True
selected_symbol_matches_active_symbol,True
outright_GC_symbol_format_only,True
contract_segment_identifiers_complete,True
tradable_flag_matches_saved_controls,True
trusted_source_still_contains_MGC,True


## 3.2 London Execution Observations

London entry eligibility is the half-open New York-time interval `[03:00, 06:00)`. The helper below uses the timezone-aware `ts_event_ny` clock fields, so 03:00 is included, 06:00 is excluded, and daylight-saving changes are handled by `America/New_York` rather than a fixed UTC offset.

In [12]:
LONDON_START_MINUTE_NY = 3 * 60
LONDON_END_MINUTE_NY = 6 * 60


def minute_of_day_ny(timestamps_ny: pd.Series) -> pd.Series:
    """Return the New York wall-clock minute for timezone-aware timestamps."""
    return timestamps_ny.dt.hour * 60 + timestamps_ny.dt.minute


def london_entry_mask(timestamps_ny: pd.Series) -> pd.Series:
    entry_minute = minute_of_day_ny(timestamps_ny)
    return entry_minute.ge(LONDON_START_MINUTE_NY) & entry_minute.lt(LONDON_END_MINUTE_NY)


london_probe = pd.Series(
    pd.to_datetime(["2024-01-02 02:59", "2024-01-02 03:00", "2024-01-02 05:59", "2024-01-02 06:00"])
    .tz_localize(RESEARCH_TIMEZONE_NAME),
    name="entry_timestamp_ny",
)
london_probe_result = london_entry_mask(london_probe)
assert london_probe_result.tolist() == [False, True, True, False]
display(pd.DataFrame({"entry_timestamp_ny": london_probe, "London_eligible": london_probe_result}))

,entry_timestamp_ny,London_eligible
0,2024-01-02 02:59:00-05:00,False
1,2024-01-02 03:00:00-05:00,True
2,2024-01-02 05:59:00-05:00,True
3,2024-01-02 06:00:00-05:00,False


## 3.3 New York Execution Observations

New York entry eligibility is the half-open interval `[07:00, 12:00)` in New York time. This explicitly excludes the 06:00-07:00 gap and any entry at noon or later. Session labels are assigned later from the entry bar timestamp only.

In [13]:
NEW_YORK_START_MINUTE_NY = 7 * 60
NEW_YORK_END_MINUTE_NY = 12 * 60


def new_york_entry_mask(timestamps_ny: pd.Series) -> pd.Series:
    entry_minute = minute_of_day_ny(timestamps_ny)
    return entry_minute.ge(NEW_YORK_START_MINUTE_NY) & entry_minute.lt(NEW_YORK_END_MINUTE_NY)


new_york_probe = pd.Series(
    pd.to_datetime(["2024-01-02 06:59", "2024-01-02 07:00", "2024-01-02 11:59", "2024-01-02 12:00"])
    .tz_localize(RESEARCH_TIMEZONE_NAME),
    name="entry_timestamp_ny",
)
new_york_probe_result = new_york_entry_mask(new_york_probe)
assert new_york_probe_result.tolist() == [False, True, True, False]
display(pd.DataFrame({"entry_timestamp_ny": new_york_probe, "New_York_eligible": new_york_probe_result}))

,entry_timestamp_ny,New_York_eligible
0,2024-01-02 06:59:00-05:00,False
1,2024-01-02 07:00:00-05:00,True
2,2024-01-02 11:59:00-05:00,True
3,2024-01-02 12:00:00-05:00,False


## 3.4 Next-Bar Entry Convention

Each GC row is treated as a candidate completed decision bar and paired vectorially with the next chronological GC row. A pair is structurally valid only when the entry row exists, all required identifiers are present, the UTC gap is exactly one minute, and product, active contract, instrument, continuous segment, and New York trading date remain unchanged. Both bars must also pass the saved tradability and boundary controls.

In [14]:
PAIR_REQUIRED_COLUMNS = [
    "source_row_id",
    "ts_event_utc",
    "ts_event_ny",
    "trade_date_ny",
    "product",
    "symbol",
    "active_symbol",
    "instrument_id",
    "continuous_segment_id",
    "tradable_research_flag",
    "roll_window_flag",
    "roll_window_type",
    "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag",
]
gc_entry_bar = gc_section_3.shift(-1).add_prefix("entry_")

pair_required_present = (
    gc_section_3[PAIR_REQUIRED_COLUMNS].notna().all(axis=1)
    & gc_entry_bar[[f"entry_{column}" for column in PAIR_REQUIRED_COLUMNS]].notna().all(axis=1)
)
decision_entry_gap = gc_entry_bar["entry_ts_event_utc"] - gc_section_3["ts_event_utc"]
same_contract_and_instrument = (
    gc_section_3["symbol"].eq(gc_entry_bar["entry_symbol"])
    & gc_section_3["active_symbol"].eq(gc_entry_bar["entry_active_symbol"])
    & gc_section_3["instrument_id"].eq(gc_entry_bar["entry_instrument_id"])
)
both_tradable = (
    gc_section_3["tradable_research_flag"]
    & gc_entry_bar["entry_tradable_research_flag"]
)
roll_controls_clear = (
    ~gc_section_3["roll_window_flag"]
    & ~gc_entry_bar["entry_roll_window_flag"]
    & gc_section_3["roll_window_type"].eq("none")
    & gc_entry_bar["entry_roll_window_type"].eq("none")
)
liquidity_controls_clear = (
    ~gc_section_3["low_liquidity_active_day_flag"]
    & ~gc_entry_bar["entry_low_liquidity_active_day_flag"]
    & ~gc_section_3["low_liquidity_warning_flag"]
    & ~gc_entry_bar["entry_low_liquidity_warning_flag"]
)
pair_rule_masks = {
    "next_GC_row_available": gc_entry_bar["entry_source_row_id"].notna(),
    "required_pair_fields_present": pair_required_present,
    "exact_one_minute_gap": decision_entry_gap.eq(pd.Timedelta(minutes=1)),
    "same_product": gc_section_3["product"].eq(gc_entry_bar["entry_product"]),
    "same_active_contract_and_instrument": same_contract_and_instrument,
    "same_continuous_segment": gc_section_3["continuous_segment_id"].eq(
        gc_entry_bar["entry_continuous_segment_id"]
    ),
    "same_New_York_trade_date": gc_section_3["trade_date_ny"].eq(gc_entry_bar["entry_trade_date_ny"]),
    "both_bars_tradable": both_tradable,
    "roll_controls_clear": roll_controls_clear,
    "liquidity_controls_clear": liquidity_controls_clear,
}

display(
    pd.Series(
        {name: int(mask.fillna(False).sum()) for name, mask in pair_rule_masks.items()},
        name="rows_passing_rule_independently",
    ).to_frame()
)

,rows_passing_rule_independently
next_GC_row_available,1759670
required_pair_fields_present,1759670
exact_one_minute_gap,1751042
same_product,1759670
same_active_contract_and_instrument,1759645
same_continuous_segment,1751038
same_New_York_trade_date,1758115
both_bars_tradable,1676565
roll_controls_clear,1681994
liquidity_controls_clear,1739466


## 3.5 Entry Cutoff Rules

The entry bar's New York timestamp determines both cutoff eligibility and session. The sequential funnel applies each rule only to rows surviving prior stages, so every exclusion is counted once and the final row count reconciles exactly to the GC source population. Entries at or after 12:00 are removed before the approved-window rule; the remaining off-window exclusions cover pre-03:00 time and the 06:00-07:00 gap.

In [15]:
entry_minute_of_day_ny = minute_of_day_ny(gc_entry_bar["entry_ts_event_ny"])
entry_before_noon = entry_minute_of_day_ny.lt(NEW_YORK_END_MINUTE_NY)
london_entry_candidate = london_entry_mask(gc_entry_bar["entry_ts_event_ny"])
new_york_entry_candidate = new_york_entry_mask(gc_entry_bar["entry_ts_event_ny"])
approved_entry_window = london_entry_candidate | new_york_entry_candidate

funnel_rules = [
    ("Next GC row available", pair_rule_masks["next_GC_row_available"]),
    ("Required pair fields present", pair_rule_masks["required_pair_fields_present"]),
    ("Exact consecutive one-minute pair", pair_rule_masks["exact_one_minute_gap"]),
    ("Same product", pair_rule_masks["same_product"]),
    ("Same active contract and instrument", pair_rule_masks["same_active_contract_and_instrument"]),
    ("Same continuous segment", pair_rule_masks["same_continuous_segment"]),
    ("Same New York trading date", pair_rule_masks["same_New_York_trade_date"]),
    ("Both bars tradable", pair_rule_masks["both_bars_tradable"]),
    ("Roll controls clear", pair_rule_masks["roll_controls_clear"]),
    ("Liquidity controls clear", pair_rule_masks["liquidity_controls_clear"]),
    ("Entry before noon NY", entry_before_noon),
    ("Entry in approved London or New York window", approved_entry_window),
]
remaining_mask = pd.Series(True, index=gc_section_3.index, dtype=bool)
funnel_records = [
    {"stage": "GC source decision rows", "excluded_at_stage": 0, "remaining_rows": gc_source_rows}
]
for stage, condition in funnel_rules:
    rows_before = int(remaining_mask.sum())
    remaining_mask &= condition.fillna(False)
    rows_after = int(remaining_mask.sum())
    funnel_records.append(
        {"stage": stage, "excluded_at_stage": rows_before - rows_after, "remaining_rows": rows_after}
    )

eligible_decision_mask = remaining_mask
exclusion_funnel = pd.DataFrame(funnel_records)
eligible_observation_count = int(eligible_decision_mask.sum())
assert int(exclusion_funnel["excluded_at_stage"].sum()) + eligible_observation_count == gc_source_rows
assert eligible_observation_count > 0
display(exclusion_funnel)

,stage,excluded_at_stage,remaining_rows
0,GC source decision rows,0,1759671
1,Next GC row available,1,1759670
2,Required pair fields present,0,1759670
3,Exact consecutive one-minute pair,8628,1751042
4,Same product,0,1751042
5,Same active contract and instrument,4,1751038
6,Same continuous segment,0,1751038
7,Same New York trading date,1271,1749767
8,Both bars tradable,79014,1670753
9,Roll controls clear,0,1670753


## 3.6 End-of-Day Exit Constraints

The compact observation index is created only for rows surviving the complete funnel. The mandatory exit is localized from the saved New York trading date at 15:30 `America/New_York`; `minutes_to_forced_exit` is schedule metadata only and does not inspect future bars or prices. Frozen research partitions are assigned from the New York entry date without using them to make any selection decision.

In [16]:
eligible_index = gc_section_3.index[eligible_decision_mask]


def decision_values(column: str) -> pd.Series:
    return gc_section_3.loc[eligible_index, column].reset_index(drop=True)


def entry_values(column: str) -> pd.Series:
    return gc_entry_bar.loc[eligible_index, f"entry_{column}"].reset_index(drop=True)


eligible_observations_gc = pd.DataFrame(
    {
        "observation_id": pd.array(np.arange(eligible_observation_count), dtype="int64[pyarrow]"),
        "decision_bar_id": decision_values("source_row_id").astype("int64[pyarrow]"),
        "entry_bar_id": entry_values("source_row_id").astype("int64[pyarrow]"),
        "decision_timestamp_utc": decision_values("ts_event_utc"),
        "decision_timestamp_ny": decision_values("ts_event_ny"),
        "entry_timestamp_utc": entry_values("ts_event_utc"),
        "entry_timestamp_ny": entry_values("ts_event_ny"),
        "trade_date_ny": entry_values("trade_date_ny"),
        "product": decision_values("product"),
        "symbol": decision_values("symbol"),
        "active_symbol": decision_values("active_symbol"),
        "instrument_id": decision_values("instrument_id"),
        "continuous_segment_id": decision_values("continuous_segment_id"),
        "decision_tradable_flag": decision_values("tradable_research_flag"),
        "entry_tradable_flag": entry_values("tradable_research_flag"),
        "decision_roll_window_flag": decision_values("roll_window_flag"),
        "entry_roll_window_flag": entry_values("roll_window_flag"),
        "decision_roll_window_type": decision_values("roll_window_type"),
        "entry_roll_window_type": entry_values("roll_window_type"),
        "decision_is_roll_day": decision_values("is_roll_day"),
        "entry_is_roll_day": entry_values("is_roll_day"),
        "decision_is_pre_roll_day": decision_values("is_pre_roll_day"),
        "entry_is_pre_roll_day": entry_values("is_pre_roll_day"),
        "decision_is_post_roll_day": decision_values("is_post_roll_day"),
        "entry_is_post_roll_day": entry_values("is_post_roll_day"),
        "decision_low_liquidity_flag": decision_values("low_liquidity_warning_flag"),
        "entry_low_liquidity_flag": entry_values("low_liquidity_warning_flag"),
    }
)
eligible_observations_gc["entry_hour_ny"] = (
    eligible_observations_gc["entry_timestamp_ny"].dt.hour.astype("int8[pyarrow]")
)
eligible_observations_gc["entry_minute_ny"] = (
    eligible_observations_gc["entry_timestamp_ny"].dt.minute.astype("int8[pyarrow]")
)
eligible_entry_minute = (
    eligible_observations_gc["entry_hour_ny"] * 60 + eligible_observations_gc["entry_minute_ny"]
)
eligible_observations_gc["entry_session"] = pd.Categorical(
    np.where(eligible_entry_minute.lt(LONDON_END_MINUTE_NY), "London", "New York"),
    categories=["London", "New York"],
    ordered=True,
)
eligible_observations_gc["entry_year"] = (
    eligible_observations_gc["entry_timestamp_ny"].dt.year.astype("int16[pyarrow]")
)
entry_date_ny = eligible_observations_gc["trade_date_ny"]
partition_labels = np.select(
    [
        entry_date_ny.le(pd.Timestamp("2023-12-31")).to_numpy(dtype=bool),
        entry_date_ny.between(pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31")).to_numpy(dtype=bool),
        entry_date_ny.between(pd.Timestamp("2025-01-01"), pd.Timestamp("2026-05-22")).to_numpy(dtype=bool),
    ],
    ["Development", "Validation", "Final test"],
    default="UNASSIGNED",
)
eligible_observations_gc["research_partition"] = pd.Categorical(
    partition_labels,
    categories=["Development", "Validation", "Final test"],
    ordered=True,
)
forced_exit_naive = eligible_observations_gc["trade_date_ny"] + pd.Timedelta(hours=15, minutes=30)
eligible_observations_gc["forced_exit_timestamp_ny"] = forced_exit_naive.dt.tz_localize(
    RESEARCH_TIMEZONE_NAME, ambiguous="raise", nonexistent="raise"
)
minutes_to_exit_float = (
    eligible_observations_gc["forced_exit_timestamp_ny"] - eligible_observations_gc["entry_timestamp_ny"]
).dt.total_seconds() / 60
assert np.allclose(minutes_to_exit_float.to_numpy(dtype=float), np.rint(minutes_to_exit_float.to_numpy(dtype=float)))
eligible_observations_gc["minutes_to_forced_exit"] = pd.array(
    np.rint(minutes_to_exit_float.to_numpy(dtype=float)).astype(np.int16), dtype="int16[pyarrow]"
)

OUTPUT_COLUMN_ORDER = [
    "observation_id", "decision_bar_id", "entry_bar_id",
    "decision_timestamp_utc", "decision_timestamp_ny", "entry_timestamp_utc", "entry_timestamp_ny",
    "trade_date_ny", "entry_session", "entry_hour_ny", "entry_minute_ny", "entry_year",
    "research_partition", "product", "symbol", "active_symbol", "instrument_id",
    "continuous_segment_id", "decision_tradable_flag", "entry_tradable_flag",
    "decision_roll_window_flag", "entry_roll_window_flag",
    "decision_roll_window_type", "entry_roll_window_type",
    "decision_is_roll_day", "entry_is_roll_day",
    "decision_is_pre_roll_day", "entry_is_pre_roll_day",
    "decision_is_post_roll_day", "entry_is_post_roll_day",
    "decision_low_liquidity_flag", "entry_low_liquidity_flag",
    "forced_exit_timestamp_ny", "minutes_to_forced_exit",
]
eligible_observations_gc = eligible_observations_gc.loc[:, OUTPUT_COLUMN_ORDER]
display(eligible_observations_gc.head(3))

,observation_id,decision_bar_id,entry_bar_id,decision_timestamp_utc,decision_timestamp_ny,entry_timestamp_utc,entry_timestamp_ny,trade_date_ny,entry_session,entry_hour_ny,entry_minute_ny,entry_year,research_partition,product,symbol,active_symbol,instrument_id,continuous_segment_id,decision_tradable_flag,entry_tradable_flag,decision_roll_window_flag,entry_roll_window_flag,decision_roll_window_type,entry_roll_window_type,decision_is_roll_day,entry_is_roll_day,decision_is_pre_roll_day,entry_is_pre_roll_day,decision_is_post_roll_day,entry_is_post_roll_day,decision_low_liquidity_flag,entry_low_liquidity_flag,forced_exit_timestamp_ny,minutes_to_forced_exit
0,0,419,420,2021-05-24 06:59:00+00:00,2021-05-24 02:59:00-04:00,2021-05-24 07:00:00+00:00,2021-05-24 03:00:00-04:00,2021-05-24 00:00:00,London,3,0,2021,Development,GC,GCM1,GCM1,100440,1,True,True,False,False,none,none,False,False,False,False,False,False,False,False,2021-05-24 15:30:00-04:00,750
1,1,420,421,2021-05-24 07:00:00+00:00,2021-05-24 03:00:00-04:00,2021-05-24 07:01:00+00:00,2021-05-24 03:01:00-04:00,2021-05-24 00:00:00,London,3,1,2021,Development,GC,GCM1,GCM1,100440,1,True,True,False,False,none,none,False,False,False,False,False,False,False,False,2021-05-24 15:30:00-04:00,749
2,2,421,422,2021-05-24 07:01:00+00:00,2021-05-24 03:01:00-04:00,2021-05-24 07:02:00+00:00,2021-05-24 03:02:00-04:00,2021-05-24 00:00:00,London,3,2,2021,Development,GC,GCM1,GCM1,100440,1,True,True,False,False,none,none,False,False,False,False,False,False,False,False,2021-05-24 15:30:00-04:00,748


## 3.7 Missing-Bar, Contract, Segment, and Abnormal-Day Handling

Production checks verify all locked boundaries and summarize the final population by session, year, and frozen partition. A separate synthetic table exercises the specified accept/reject edge cases without modifying production data. The saved `tradable_research_flag` exactly incorporates roll-window and liquidity exclusions; these component flags are still checked explicitly and retained for auditability.

In [17]:
session_counts = eligible_observations_gc["entry_session"].value_counts(sort=False)
year_counts = eligible_observations_gc["entry_year"].value_counts(sort=False).sort_index()
partition_counts = eligible_observations_gc["research_partition"].value_counts(sort=False)
saved_field_missingness = eligible_observations_gc.isna().sum()

important_boundary_minutes = [(2, 59), (3, 0), (5, 59), (6, 0), (6, 59), (7, 0), (11, 59), (12, 0)]
boundary_minute_counts = pd.DataFrame(
    [
        {
            "entry_time_ny": f"{hour:02d}:{minute:02d}",
            "eligible_entries": int(
                (
                    eligible_observations_gc["entry_hour_ny"].eq(hour)
                    & eligible_observations_gc["entry_minute_ny"].eq(minute)
                ).sum()
            ),
        }
        for hour, minute in important_boundary_minutes
    ]
)

eligible_pair_rule_checks = {
    name: bool(mask.loc[eligible_index].all()) for name, mask in pair_rule_masks.items()
}
production_validation_checks = pd.Series(
    {
        "expected_eligible_row_count": len(eligible_observations_gc) == eligible_observation_count,
        "all_pair_rules_pass": all(eligible_pair_rule_checks.values()),
        "entry_strictly_after_decision": bool(
            eligible_observations_gc["entry_timestamp_utc"].gt(
                eligible_observations_gc["decision_timestamp_utc"]
            ).all()
        ),
        "decision_entry_gap_exactly_one_minute": bool(
            eligible_observations_gc["entry_timestamp_utc"]
            .sub(eligible_observations_gc["decision_timestamp_utc"])
            .eq(pd.Timedelta(minutes=1))
            .all()
        ),
        "entry_timestamps_sorted": eligible_observations_gc["entry_timestamp_utc"].is_monotonic_increasing,
        "London_window_exact": bool(
            eligible_entry_minute[eligible_observations_gc["entry_session"].eq("London")]
            .between(LONDON_START_MINUTE_NY, LONDON_END_MINUTE_NY - 1)
            .all()
        ),
        "New_York_window_exact": bool(
            eligible_entry_minute[eligible_observations_gc["entry_session"].eq("New York")]
            .between(NEW_YORK_START_MINUTE_NY, NEW_YORK_END_MINUTE_NY - 1)
            .all()
        ),
        "no_entries_during_06_to_07_gap": bool(
            not eligible_entry_minute.between(LONDON_END_MINUTE_NY, NEW_YORK_START_MINUTE_NY - 1).any()
        ),
        "no_entries_at_or_after_noon": bool(eligible_entry_minute.lt(NEW_YORK_END_MINUTE_NY).all()),
        "same_New_York_date_as_forced_exit": bool(
            eligible_observations_gc["forced_exit_timestamp_ny"].dt.date.eq(
                eligible_observations_gc["trade_date_ny"].dt.date
            ).all()
        ),
        "minutes_to_forced_exit_positive": bool(
            eligible_observations_gc["minutes_to_forced_exit"].gt(0).all()
        ),
        "observation_ids_unique": eligible_observations_gc["observation_id"].is_unique,
        "decision_entry_pairs_unique": not eligible_observations_gc.duplicated(
            ["decision_bar_id", "entry_bar_id"]
        ).any(),
        "all_saved_fields_complete": int(saved_field_missingness.sum()) == 0,
        "forward_columns_absent": not set(EXISTING_FORWARD_COLUMNS).intersection(
            eligible_observations_gc.columns
        ),
        "all_partitions_assigned": "UNASSIGNED" not in set(partition_labels),
        "all_decision_and_entry_bars_tradable": bool(
            eligible_observations_gc[["decision_tradable_flag", "entry_tradable_flag"]]
            .all(axis=None)
        ),
        "all_roll_and_liquidity_controls_clear": bool(
            ~eligible_observations_gc[
                ["decision_roll_window_flag", "entry_roll_window_flag",
                 "decision_low_liquidity_flag", "entry_low_liquidity_flag"]
            ].any(axis=None)
        ),
    },
    name="passed",
)

display(pd.Series({"GC_source_rows": gc_source_rows, "eligible_observations": len(eligible_observations_gc),
                   "first_entry_ny": eligible_observations_gc["entry_timestamp_ny"].min(),
                   "last_entry_ny": eligible_observations_gc["entry_timestamp_ny"].max()}, name="value").to_frame())
display(session_counts.rename("eligible_observations").to_frame())
display(year_counts.rename("eligible_observations").to_frame())
display(partition_counts.rename("eligible_observations").to_frame())
display(boundary_minute_counts)
display(saved_field_missingness.rename("missing_values").to_frame())
display(production_validation_checks.to_frame())
assert production_validation_checks.all(), production_validation_checks[~production_validation_checks].index.tolist()

,value
GC_source_rows,1759671
eligible_observations,586530
first_entry_ny,2021-05-24 03:00:00-04:00
last_entry_ny,2026-05-22 11:59:00-04:00


,eligible_observations
entry_session,
London,219938
New York,366592


,eligible_observations
entry_year,
2021,72960
2022,117114
2023,116156
2024,118076
2025,117592
2026,44632


,eligible_observations
research_partition,
Development,306230
Validation,118076
Final test,162224


,entry_time_ny,eligible_entries
0,02:59,0
1,03:00,1222
2,05:59,1222
3,06:00,0
4,06:59,0
5,07:00,1222
6,11:59,1222
7,12:00,0


,missing_values
observation_id,0
decision_bar_id,0
entry_bar_id,0
decision_timestamp_utc,0
decision_timestamp_ny,0
entry_timestamp_utc,0
entry_timestamp_ny,0
trade_date_ny,0
entry_session,0
entry_hour_ny,0


,passed
expected_eligible_row_count,True
all_pair_rules_pass,True
entry_strictly_after_decision,True
decision_entry_gap_exactly_one_minute,True
entry_timestamps_sorted,True
London_window_exact,True
New_York_window_exact,True
no_entries_during_06_to_07_gap,True
no_entries_at_or_after_noon,True
same_New_York_date_as_forced_exit,True


In [18]:
synthetic_pairs = pd.DataFrame(
    {
        "case": [
            "02:59 to 03:00", "05:59 to 06:00", "06:59 to 07:00", "11:58 to 11:59",
            "11:59 to 12:00", "contract change", "non-consecutive pair", "New York date crossing",
        ],
        "decision_time": [
            "2024-01-02 02:59", "2024-01-02 05:59", "2024-01-02 06:59", "2024-01-02 11:58",
            "2024-01-02 11:59", "2024-01-02 07:00", "2024-01-02 07:00", "2024-01-02 23:59",
        ],
        "entry_time": [
            "2024-01-02 03:00", "2024-01-02 06:00", "2024-01-02 07:00", "2024-01-02 11:59",
            "2024-01-02 12:00", "2024-01-02 07:01", "2024-01-02 07:02", "2024-01-03 00:00",
        ],
        "decision_symbol": ["GCG4"] * 8,
        "entry_symbol": ["GCG4", "GCG4", "GCG4", "GCG4", "GCG4", "GCJ4", "GCG4", "GCG4"],
        "expected_eligible": [True, False, True, True, False, False, False, False],
    }
)
synthetic_pairs["decision_timestamp_ny"] = pd.to_datetime(synthetic_pairs.pop("decision_time")).dt.tz_localize(
    RESEARCH_TIMEZONE_NAME
)
synthetic_pairs["entry_timestamp_ny"] = pd.to_datetime(synthetic_pairs.pop("entry_time")).dt.tz_localize(
    RESEARCH_TIMEZONE_NAME
)
synthetic_gap_valid = synthetic_pairs["entry_timestamp_ny"].sub(
    synthetic_pairs["decision_timestamp_ny"]
).eq(pd.Timedelta(minutes=1))
synthetic_contract_valid = synthetic_pairs["decision_symbol"].eq(synthetic_pairs["entry_symbol"])
synthetic_date_valid = synthetic_pairs["decision_timestamp_ny"].dt.date.eq(
    synthetic_pairs["entry_timestamp_ny"].dt.date
)
synthetic_window_valid = london_entry_mask(synthetic_pairs["entry_timestamp_ny"]) | new_york_entry_mask(
    synthetic_pairs["entry_timestamp_ny"]
)
synthetic_pairs["actual_eligible"] = (
    synthetic_gap_valid & synthetic_contract_valid & synthetic_date_valid & synthetic_window_valid
)
synthetic_pairs["test_passed"] = synthetic_pairs["actual_eligible"].eq(
    synthetic_pairs["expected_eligible"]
)
display(synthetic_pairs[["case", "decision_timestamp_ny", "entry_timestamp_ny",
                         "expected_eligible", "actual_eligible", "test_passed"]])
assert synthetic_pairs["test_passed"].all()

,case,decision_timestamp_ny,entry_timestamp_ny,expected_eligible,actual_eligible,test_passed
0,02:59 to 03:00,2024-01-02 02:59:00-05:00,2024-01-02 03:00:00-05:00,True,True,True
1,05:59 to 06:00,2024-01-02 05:59:00-05:00,2024-01-02 06:00:00-05:00,False,False,True
2,06:59 to 07:00,2024-01-02 06:59:00-05:00,2024-01-02 07:00:00-05:00,True,True,True
3,11:58 to 11:59,2024-01-02 11:58:00-05:00,2024-01-02 11:59:00-05:00,True,True,True
4,11:59 to 12:00,2024-01-02 11:59:00-05:00,2024-01-02 12:00:00-05:00,False,False,True
5,contract change,2024-01-02 07:00:00-05:00,2024-01-02 07:01:00-05:00,False,False,True
6,non-consecutive pair,2024-01-02 07:00:00-05:00,2024-01-02 07:02:00-05:00,False,False,True
7,New York date crossing,2024-01-02 23:59:00-05:00,2024-01-03 00:00:00-05:00,False,False,True


## 3.8 Save Eligible Observation Index

The validated index is saved under the generated-data directory with Zstandard compression, then reloaded as an independent artifact. Reload checks confirm row and column identity, timestamp timezone preservation, identifier uniqueness, complete saved fields, and continued exclusion of every quarantined forward-looking column. The generated parquet is excluded from Git by the project data ignore rule.

In [19]:
STATISTICAL_RESEARCH_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "statistical_research"
ELIGIBLE_OBSERVATIONS_PATH = STATISTICAL_RESEARCH_DATA_DIR / "eligible_observations_gc.parquet"
STATISTICAL_RESEARCH_DATA_DIR.mkdir(parents=True, exist_ok=True)
eligible_observations_gc.to_parquet(
    ELIGIBLE_OBSERVATIONS_PATH, engine="pyarrow", compression="zstd", index=False
)

reloaded_eligible_observations_gc = pd.read_parquet(
    ELIGIBLE_OBSERVATIONS_PATH, engine="pyarrow", dtype_backend="pyarrow"
)
reloaded_schema = pq.ParquetFile(ELIGIBLE_OBSERVATIONS_PATH).schema_arrow
reload_missingness = reloaded_eligible_observations_gc.isna().sum()
reload_checks = pd.Series(
    {
        "row_count_preserved": len(reloaded_eligible_observations_gc) == len(eligible_observations_gc),
        "column_order_preserved": list(reloaded_eligible_observations_gc.columns) == OUTPUT_COLUMN_ORDER,
        "decision_UTC_timezone_preserved": reloaded_schema.field("decision_timestamp_utc").type.tz == "UTC",
        "entry_UTC_timezone_preserved": reloaded_schema.field("entry_timestamp_utc").type.tz == "UTC",
        "decision_New_York_timezone_preserved": (
            reloaded_schema.field("decision_timestamp_ny").type.tz == RESEARCH_TIMEZONE_NAME
        ),
        "entry_New_York_timezone_preserved": (
            reloaded_schema.field("entry_timestamp_ny").type.tz == RESEARCH_TIMEZONE_NAME
        ),
        "forced_exit_New_York_timezone_preserved": (
            reloaded_schema.field("forced_exit_timestamp_ny").type.tz == RESEARCH_TIMEZONE_NAME
        ),
        "observation_ids_unique_after_reload": reloaded_eligible_observations_gc["observation_id"].is_unique,
        "decision_entry_pairs_unique_after_reload": not reloaded_eligible_observations_gc.duplicated(
            ["decision_bar_id", "entry_bar_id"]
        ).any(),
        "no_missing_saved_values_after_reload": int(reload_missingness.sum()) == 0,
        "no_quarantined_forward_columns_saved": not set(EXISTING_FORWARD_COLUMNS).intersection(
            reloaded_eligible_observations_gc.columns
        ),
    },
    name="passed",
)
display(pd.DataFrame({"column": reloaded_schema.names,
                      "saved_arrow_type": [str(field.type) for field in reloaded_schema]}))
display(reload_checks.to_frame())
assert reload_checks.all(), reload_checks[~reload_checks].index.tolist()

eligible_observations_gc = reloaded_eligible_observations_gc
section_3_ready = bool(production_validation_checks.all() and synthetic_pairs["test_passed"].all() and reload_checks.all())
print(f"Final eligibility: valid GC decision bar t mapped to the true one-minute entry bar t+1; both bars are "
      f"tradable, roll/liquidity clear, contract/instrument/segment/date stable, with entry in [03:00, 06:00) "
      f"or [07:00, 12:00) America/New_York.")
print(f"Eligible observations: {len(eligible_observations_gc):,}")
print(f"Session split: {session_counts.to_dict()}")
print(f"Partition split: {partition_counts.to_dict()}")
print(f"Saved output: {ELIGIBLE_OBSERVATIONS_PATH.relative_to(PROJECT_ROOT)}")
print("Schema fallback: none; symbol, active_symbol, instrument_id, and continuous_segment_id were available.")
print("Section 4 is ready to begin, but no Section 4 work is performed here.")
print("SECTION 3 STATUS: READY" if section_3_ready else "SECTION 3 STATUS: NOT READY")
assert section_3_ready

del gc_entry_bar, remaining_mask, eligible_decision_mask, forced_exit_naive, minutes_to_exit_float

,column,saved_arrow_type
0,observation_id,int64
1,decision_bar_id,int64
2,entry_bar_id,int64
3,decision_timestamp_utc,"timestamp[ns, tz=UTC]"
4,decision_timestamp_ny,"timestamp[ns, tz=America/New_York]"
5,entry_timestamp_utc,"timestamp[ns, tz=UTC]"
6,entry_timestamp_ny,"timestamp[ns, tz=America/New_York]"
7,trade_date_ny,timestamp[ms]
8,entry_session,"dictionary<values=string, indices=int8, ordere..."
9,entry_hour_ny,int8


,passed
row_count_preserved,True
column_order_preserved,True
decision_UTC_timezone_preserved,True
entry_UTC_timezone_preserved,True
decision_New_York_timezone_preserved,True
entry_New_York_timezone_preserved,True
forced_exit_New_York_timezone_preserved,True
observation_ids_unique_after_reload,True
decision_entry_pairs_unique_after_reload,True
no_missing_saved_values_after_reload,True


Final eligibility: valid GC decision bar t mapped to the true one-minute entry bar t+1; both bars are tradable, roll/liquidity clear, contract/instrument/segment/date stable, with entry in [03:00, 06:00) or [07:00, 12:00) America/New_York.
Eligible observations: 586,530
Session split: {'London': 219938, 'New York': 366592}
Partition split: {'Development': 306230, 'Validation': 118076, 'Final test': 162224}
Saved output: data\processed\statistical_research\eligible_observations_gc.parquet
Schema fallback: none; symbol, active_symbol, instrument_id, and continuous_segment_id were available.
Section 4 is ready to begin, but no Section 4 work is performed here.
SECTION 3 STATUS: READY


# 4.0 Construct Forward Outcome Labels

Section 4 preserves all 586,530 Section 3 observations and measures future GC market outcomes without POI inputs, stops, targets, trade simulation, or R multiples. For horizon `h`, the decision uses completed bar `t`, entry is the open of bar `t+1`, and the path contains exactly `h` consecutive one-minute bars beginning with the entry bar. Thus the final source bar starts at `entry + (h - 1) minutes`, its close is the exit price, and the economic exit timestamp saved below is `entry + h minutes`. A requested horizon is never shortened.

GC prices must first pass a strict 0.10 tick-grid check. ATR-normalized outcomes use only `rolling_atr_20m` from decision bar `t`, saved as `decision_atr_20m`; missing or nonpositive ATR nulls only normalized outcomes and expansion, not otherwise valid raw paths.

In [20]:
import io
import unittest

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.statistical_research.labels import (
    EXPANSION_QUANTILE,
    FORWARD_HORIZONS_MINUTES,
    GC_TICK_SIZE,
    LABEL_REASON_CATEGORIES,
    build_forward_label_table,
    build_label_availability_report,
)

assert section_3_ready
assert FORWARD_HORIZONS_MINUTES == (5, 15, 30, 60, 120, 180)
assert GC_TICK_SIZE == 0.10
assert EXPANSION_QUANTILE == 0.80
EXPECTED_SECTION_4_ROWS = 586_530
LABEL_CHUNK_SIZE = 20_000
FORWARD_LABELS_PATH = STATISTICAL_RESEARCH_DATA_DIR / "forward_labels_gc.parquet"
EXPANSION_THRESHOLDS_PATH = STATISTICAL_RESEARCH_DATA_DIR / "forward_label_expansion_thresholds_gc.parquet"

SECTION_4_SOURCE_COLUMNS = [
    "ts_event_utc", "ts_event_ny", "trade_date_ny", "product", "symbol",
    "active_symbol", "instrument_id", "continuous_segment_id",
    "open", "high", "low", "close", "tradable_research_flag",
    "roll_window_flag", "roll_window_type", "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag", "rolling_atr_20m",
]
TRUSTED_SAMPLE_INDEX = np.linspace(0, len(research_bars) - 1, 11, dtype=np.int64)
trusted_research_bars_signature = {
    "object_id": id(research_bars),
    "shape": research_bars.shape,
    "columns": tuple(research_bars.columns),
    "dtypes": tuple(map(str, research_bars.dtypes)),
    "sample_hash": int(pd.util.hash_pandas_object(research_bars.iloc[TRUSTED_SAMPLE_INDEX], index=True).sum()),
}
gc_path_source = research_bars.loc[gc_source_mask, SECTION_4_SOURCE_COLUMNS].copy()
gc_path_source.insert(0, "source_row_id", pd.array(gc_source_row_ids, dtype="int64[pyarrow]"))
assert len(gc_path_source) == EXPECTED_ROW_COUNTS[PRIMARY_RESEARCH_PRODUCT]
assert gc_path_source["source_row_id"].is_unique and gc_path_source["source_row_id"].is_monotonic_increasing
assert set(gc_path_source["product"].unique().tolist()) == {"GC"}
display(pd.Series({"GC_path_source_rows": len(gc_path_source),
                   "path_source_columns": gc_path_source.shape[1],
                   "bounded_extrema_chunk_rows": LABEL_CHUNK_SIZE}, name="value").to_frame())

,value
GC_path_source_rows,1759671
path_source_columns,19
bounded_extrema_chunk_rows,20000


## 4.1 Forward Returns

For each complete fixed path, signed arithmetic movement is measured from the entry-bar open to the final path-bar close. Outputs are stored in GC ticks, arithmetic basis points, and decision-bar ATR units. Positive is upward, negative is downward, and zero is unchanged. `exit_timestamp_*_<h>` is the economic close timestamp (`entry + h`), not the timestamp at which the final source bar opened.

In [21]:
label_build_result = build_forward_label_table(
    eligible_observations_gc,
    gc_path_source,
    horizons=FORWARD_HORIZONS_MINUTES,
    tick_size=GC_TICK_SIZE,
    expansion_quantile=EXPANSION_QUANTILE,
    chunk_size=LABEL_CHUNK_SIZE,
)
forward_labels_gc = label_build_result.labels
expansion_thresholds_gc = label_build_result.expansion_thresholds
tick_grid_validation = label_build_result.tick_grid_validation
FORWARD_LABEL_COLUMN_ORDER = list(label_build_result.column_order)

horizon_availability_counts = pd.DataFrame(
    [
        {
            "horizon": h,
            "available_observations": int(forward_labels_gc[f"label_available_{h}"].sum()),
            "unavailable_observations": int((~forward_labels_gc[f"label_available_{h}"]).sum()),
            "availability_rate": float(forward_labels_gc[f"label_available_{h}"].mean()),
        }
        for h in FORWARD_HORIZONS_MINUTES
    ]
)
display(tick_grid_validation)
display(horizon_availability_counts)
display(forward_labels_gc[["observation_id", "entry_timestamp_ny", "entry_price",
                           "decision_atr_20m", "atr_normalization_available",
                           "forward_return_5_ticks", "forward_return_5_bps",
                           "forward_return_5_atr"]].head(3))

,field,finite_prices_checked,tick_size,tolerance_ticks,maximum_grid_deviation_ticks,off_grid_prices
0,open,1759671,0.100000,0.000000,0.000000,0
1,high,1759671,0.100000,0.000000,0.000000,0
2,low,1759671,0.100000,0.000000,0.000000,0
3,close,1759671,0.100000,0.000000,0.000000,0


,horizon,available_observations,unavailable_observations,availability_rate
0,5,586468,62,0.999894
1,15,586296,234,0.999601
2,30,585958,572,0.999025
3,60,585033,1497,0.997448
4,120,582471,4059,0.993080
5,180,579166,7364,0.987445


,observation_id,entry_timestamp_ny,entry_price,decision_atr_20m,atr_normalization_available,forward_return_5_ticks,forward_return_5_bps,forward_return_5_atr
0,0,2021-05-24 03:00:00-04:00,"1,882.200000",0.435000,True,3,1.593880,0.689655
1,1,2021-05-24 03:01:00-04:00,"1,882.400000",0.450000,True,3,1.593710,0.666667
2,2,2021-05-24 03:02:00-04:00,"1,881.700000",0.485000,True,14,7.440081,2.886598


## 4.2 Maximum Favourable Excursion

Long MFE is the path maximum high minus entry price; short MFE is entry price minus the path minimum low. Tick magnitudes are nonnegative and ATR units use decision-bar ATR. Time to MFE is one-based and records the first path minute at which the final extreme occurs, including minute 1 when reached in the entry bar. Repeated equal extrema use their first occurrence.

## 4.3 Maximum Adverse Excursion

Long MAE is entry price minus the path minimum low; short MAE is the path maximum high minus entry price. Adverse excursion is stored as a positive magnitude, with the same one-based first-occurrence timing convention. Because one-minute OHLC does not reveal whether a bar's high or low occurred first, Section 4 makes no within-bar ordering claim and creates no MFE-before-MAE label.

## 4.4 Future Range and Volatility

Future range is path maximum high minus path minimum low, in ticks and decision ATR units. Realized volatility is unannualized total path variation: the first log return is `log(entry-bar close / entry open)`, later returns are consecutive path-close log returns, and the label is `sqrt(sum(r_i^2)) × 10,000` basis points. It excludes the decision-close-to-entry-open movement and is neither annualized nor a sample standard deviation.

## 4.5 Directional and Expansion Labels

Direction is nullable signed integer `-1`, `0`, or `+1` from the raw tick return, with no arbitrary cost threshold. Expansion is nullable boolean and compares `future_range_<h>_atr` with one frozen per-horizon 80th-percentile threshold fitted across label-available Development observations with valid decision ATR. Validation and Final test never enter threshold fitting; the frozen thresholds are applied unchanged to all partitions.

In [22]:
atr_normalization_coverage = pd.Series(
    {
        "available_observations": int(forward_labels_gc["atr_normalization_available"].sum()),
        "unavailable_observations": int((~forward_labels_gc["atr_normalization_available"]).sum()),
        "coverage_rate": float(forward_labels_gc["atr_normalization_available"].mean()),
    },
    name="value",
)
display(atr_normalization_coverage.to_frame())
display(expansion_thresholds_gc)

,value
available_observations,"586,382.000000"
unavailable_observations,148.000000
coverage_rate,0.999748


,horizon,quantile,threshold,development_observation_count_used,fitting_partition,normalization_denominator
0,5,0.800000,3.111111,306161,Development,decision_atr_20m
1,15,0.800000,5.557772,306108,Development,decision_atr_20m
2,30,0.800000,8.000000,305979,Development,decision_atr_20m
3,60,0.800000,11.797753,305603,Development,decision_atr_20m
4,120,0.800000,17.413793,304370,Development,decision_atr_20m
5,180,0.800000,22.409639,302774,Development,decision_atr_20m


## 4.6 Label Availability and End-of-Day Rules

Every horizon receives an explicit availability flag and an ordered categorical first-failure reason. Priority is: insufficient future rows; missing/nonconsecutive minute; contract or instrument change; continuous-segment change; New York date change; tradability, roll, or liquidity failure; forced-exit breach; invalid price data. A valid final bar must close no later than 15:30 New York (`final source timestamp + 1 minute <= forced exit`). Paths may continue beyond their London or New York entry window. Unavailable paths have null path outputs, while all original observations remain in the table.

In [23]:
synthetic_test_stream = io.StringIO()
synthetic_test_suite = unittest.defaultTestLoader.loadTestsFromName(
    "tests.test_statistical_research_labels"
)
synthetic_test_result = unittest.TextTestRunner(
    stream=synthetic_test_stream, verbosity=2
).run(synthetic_test_suite)
print(synthetic_test_stream.getvalue())
print(f"Synthetic tests passed: {synthetic_test_result.testsRun - len(synthetic_test_result.failures) - len(synthetic_test_result.errors)}/{synthetic_test_result.testsRun}")
assert synthetic_test_result.testsRun >= 20
assert synthetic_test_result.wasSuccessful()

test_continuous_segment_change (tests.test_statistical_research_labels.ForwardLabelAvailabilityTests.test_continuous_segment_change) ... ok
test_contract_or_instrument_change (tests.test_statistical_research_labels.ForwardLabelAvailabilityTests.test_contract_or_instrument_change) ... ok
test_exact_forced_exit_boundary_is_available (tests.test_statistical_research_labels.ForwardLabelAvailabilityTests.test_exact_forced_exit_boundary_is_available) ... ok
test_first_failure_priority_is_deterministic (tests.test_statistical_research_labels.ForwardLabelAvailabilityTests.test_first_failure_priority_is_deterministic) ... ok
test_incomplete_180_minute_path_is_not_shortened (tests.test_statistical_research_labels.ForwardLabelAvailabilityTests.test_incomplete_180_minute_path_is_not_shortened) ... ok
test_invalid_price_inside_path (tests.test_statistical_research_labels.ForwardLabelAvailabilityTests.test_invalid_price_inside_path) ... ok
test_missing_one_minute_bar (tests.test_statistical_research

In [24]:
def epoch_nanoseconds(values: pd.Series) -> np.ndarray:
    return pd.to_datetime(values, utc=True).to_numpy(dtype="datetime64[ns]").view(np.int64)


def prefix_counts(flags: np.ndarray) -> np.ndarray:
    prefix = np.zeros(len(flags) + 1, dtype=np.int64)
    np.cumsum(flags, dtype=np.int64, out=prefix[1:])
    return prefix


def validate_label_invariants(frame: pd.DataFrame) -> pd.Series:
    checks = {
        "exactly_one_row_per_observation": len(frame) == EXPECTED_SECTION_4_ROWS and frame["observation_id"].is_unique,
        "observation_ids_ordered": frame["observation_id"].is_monotonic_increasing,
        "observation_ids_match_section_3": np.array_equal(frame["observation_id"], eligible_observations_gc["observation_id"]),
        "entry_ids_match_section_3": np.array_equal(frame["entry_bar_id"], eligible_observations_gc["entry_bar_id"]),
        "entry_timestamps_match_section_3": np.array_equal(epoch_nanoseconds(frame["entry_timestamp_utc"]), epoch_nanoseconds(eligible_observations_gc["entry_timestamp_utc"])),
        "column_order_stable": list(frame.columns) == FORWARD_LABEL_COLUMN_ORDER,
    }
    prior_available = np.ones(len(frame), dtype=bool)
    atr_ok = frame["atr_normalization_available"].to_numpy(dtype=bool)
    for h in FORWARD_HORIZONS_MINUTES:
        available = frame[f"label_available_{h}"].to_numpy(dtype=bool)
        unavailable = ~available
        raw_columns = [
            f"exit_timestamp_utc_{h}", f"exit_timestamp_ny_{h}", f"exit_price_{h}",
            f"forward_return_{h}_ticks", f"forward_return_{h}_bps",
            f"mfe_long_{h}_ticks", f"mfe_short_{h}_ticks",
            f"mae_long_{h}_ticks", f"mae_short_{h}_ticks",
            f"time_to_mfe_long_{h}_minutes", f"time_to_mfe_short_{h}_minutes",
            f"time_to_mae_long_{h}_minutes", f"time_to_mae_short_{h}_minutes",
            f"future_range_{h}_ticks", f"future_realized_volatility_{h}_bps",
            f"direction_label_{h}",
        ]
        normalized_columns = [
            f"forward_return_{h}_atr", f"mfe_long_{h}_atr", f"mfe_short_{h}_atr",
            f"mae_long_{h}_atr", f"mae_short_{h}_atr", f"future_range_{h}_atr",
        ]
        expected_normalized = available & atr_ok
        ticks = {name: frame[f"{name}_{h}_ticks"].to_numpy(dtype=np.int64, na_value=0)
                 for name in ("mfe_long", "mfe_short", "mae_long", "mae_short", "future_range")}
        forward_ticks = frame[f"forward_return_{h}_ticks"].to_numpy(dtype=np.int64, na_value=0)
        rv = frame[f"future_realized_volatility_{h}_bps"].to_numpy(dtype=float, na_value=np.nan)
        times = frame[[f"time_to_mfe_long_{h}_minutes", f"time_to_mfe_short_{h}_minutes",
                       f"time_to_mae_long_{h}_minutes", f"time_to_mae_short_{h}_minutes"]].to_numpy(dtype=np.int16, na_value=0)
        reason_available = frame[f"label_unavailable_reason_{h}"].astype(str).eq("available").to_numpy()
        checks.update({
            f"{h}_availability_nonincreasing": bool((~available | prior_available).all()),
            f"{h}_availability_reason_consistent": bool(np.array_equal(available, reason_available)),
            f"{h}_available_raw_complete": bool(frame.loc[available, raw_columns].notna().all(axis=None)),
            f"{h}_unavailable_raw_null": bool(frame.loc[unavailable, raw_columns].isna().all(axis=None)),
            f"{h}_ATR_null_semantics": all(np.array_equal(frame[column].notna().to_numpy(), expected_normalized) for column in normalized_columns),
            f"{h}_expansion_null_semantics": bool(np.array_equal(frame[f"expansion_label_{h}"].notna().to_numpy(), expected_normalized)),
            f"{h}_excursions_nonnegative": all(bool((values[available] >= 0).all()) for values in ticks.values()),
            f"{h}_long_short_symmetry": bool(np.array_equal(ticks["mfe_long"][available], ticks["mae_short"][available]) and np.array_equal(ticks["mae_long"][available], ticks["mfe_short"][available])),
            f"{h}_range_identity": bool(np.array_equal(ticks["future_range"][available], ticks["mfe_long"][available] + ticks["mae_long"][available]) and np.array_equal(ticks["future_range"][available], ticks["mfe_short"][available] + ticks["mae_short"][available])),
            f"{h}_return_inside_excursions": bool(((forward_ticks[available] >= -ticks["mae_long"][available]) & (forward_ticks[available] <= ticks["mfe_long"][available])).all()),
            f"{h}_time_to_extreme_bounds": bool(((times[available] >= 1) & (times[available] <= h)).all()),
            f"{h}_realized_volatility_valid": bool((np.isfinite(rv[available]) & (rv[available] >= 0.0)).all()),
            f"{h}_direction_domain": set(frame[f"direction_label_{h}"].dropna().astype("int8").unique().tolist()).issubset({-1, 0, 1}),
        })
        prior_available = available
    return pd.Series(checks, name="passed", dtype=bool)


source_ids = gc_path_source["source_row_id"].to_numpy(dtype=np.int64)
entry_ids = forward_labels_gc["entry_bar_id"].to_numpy(dtype=np.int64)
entry_positions = np.searchsorted(source_ids, entry_ids)
assert np.array_equal(source_ids[entry_positions], entry_ids)
source_ts_ns = epoch_nanoseconds(gc_path_source["ts_event_utc"])
source_symbols = gc_path_source["symbol"].astype(str).to_numpy()
source_active_symbols = gc_path_source["active_symbol"].astype(str).to_numpy()
source_instruments = gc_path_source["instrument_id"].to_numpy(dtype=np.int64)
source_segments = gc_path_source["continuous_segment_id"].to_numpy(dtype=np.int64)
source_dates = pd.to_datetime(gc_path_source["trade_date_ny"]).to_numpy(dtype="datetime64[D]")
source_ohlc = gc_path_source[["open", "high", "low", "close"]].to_numpy(dtype=float, na_value=np.nan)
source_price_valid = (np.isfinite(source_ohlc).all(axis=1) & (source_ohlc > 0.0).all(axis=1) &
                      (source_ohlc[:, 1] >= np.maximum(source_ohlc[:, 0], source_ohlc[:, 3])) &
                      (source_ohlc[:, 2] <= np.minimum(source_ohlc[:, 0], source_ohlc[:, 3])))
source_control_valid = (
    gc_path_source["tradable_research_flag"].to_numpy(dtype=bool)
    & ~gc_path_source["roll_window_flag"].to_numpy(dtype=bool)
    & gc_path_source["roll_window_type"].eq("none").to_numpy(dtype=bool)
    & ~gc_path_source["low_liquidity_active_day_flag"].to_numpy(dtype=bool)
    & ~gc_path_source["low_liquidity_warning_flag"].to_numpy(dtype=bool)
)
transition_time_bad = np.r_[False, np.diff(source_ts_ns) != 60_000_000_000]
transition_contract_bad = np.r_[False, (source_symbols[1:] != source_symbols[:-1]) |
    (source_active_symbols[1:] != source_active_symbols[:-1]) | (source_instruments[1:] != source_instruments[:-1])]
transition_segment_bad = np.r_[False, source_segments[1:] != source_segments[:-1]]
transition_date_bad = np.r_[False, source_dates[1:] != source_dates[:-1]]
prefix_time_bad = prefix_counts(transition_time_bad)
prefix_contract_bad = prefix_counts(transition_contract_bad)
prefix_segment_bad = prefix_counts(transition_segment_bad)
prefix_date_bad = prefix_counts(transition_date_bad)
prefix_control_bad = prefix_counts(~source_control_valid)
prefix_price_bad = prefix_counts(~source_price_valid)

source_boundary_checks = {}
entry_utc_ns = epoch_nanoseconds(forward_labels_gc["entry_timestamp_utc"])
forced_exit_ns = epoch_nanoseconds(forward_labels_gc["forced_exit_timestamp_ny"])
source_closes = gc_path_source["close"].to_numpy(dtype=float, na_value=np.nan)
for h in FORWARD_HORIZONS_MINUTES:
    available = forward_labels_gc[f"label_available_{h}"].to_numpy(dtype=bool)
    rows = np.flatnonzero(available)
    starts = entry_positions[rows]
    ends_exclusive = starts + h
    ends = ends_exclusive - 1
    exit_ns = epoch_nanoseconds(forward_labels_gc.loc[available, f"exit_timestamp_utc_{h}"])
    source_boundary_checks.update({
        f"{h}_exact_requested_bar_count": bool((ends_exclusive <= len(gc_path_source)).all()),
        f"{h}_exact_final_source_timestamp": bool(np.array_equal(source_ts_ns[ends], entry_utc_ns[rows] + (h - 1) * 60_000_000_000)),
        f"{h}_exact_economic_exit_timestamp": bool(np.array_equal(exit_ns, entry_utc_ns[rows] + h * 60_000_000_000)),
        f"{h}_no_missing_minute": bool(((prefix_time_bad[ends_exclusive] - prefix_time_bad[starts + 1]) == 0).all()),
        f"{h}_no_contract_or_instrument_change": bool(((prefix_contract_bad[ends_exclusive] - prefix_contract_bad[starts + 1]) == 0).all()),
        f"{h}_no_segment_change": bool(((prefix_segment_bad[ends_exclusive] - prefix_segment_bad[starts + 1]) == 0).all()),
        f"{h}_no_New_York_date_change": bool(((prefix_date_bad[ends_exclusive] - prefix_date_bad[starts + 1]) == 0).all()),
        f"{h}_controls_valid_through_path": bool(((prefix_control_bad[ends_exclusive] - prefix_control_bad[starts]) == 0).all()),
        f"{h}_prices_valid_through_path": bool(((prefix_price_bad[ends_exclusive] - prefix_price_bad[starts]) == 0).all()),
        f"{h}_forced_exit_not_breached": bool((exit_ns <= forced_exit_ns[rows]).all()),
        f"{h}_exit_price_matches_final_close": bool(np.allclose(forward_labels_gc.loc[available, f"exit_price_{h}"].to_numpy(dtype=float), source_closes[ends], rtol=0.0, atol=1e-12)),
    })

threshold_checks = {}
development_mask = forward_labels_gc["research_partition"].eq("Development").to_numpy(dtype=bool)
atr_mask = forward_labels_gc["atr_normalization_available"].to_numpy(dtype=bool)
for h in FORWARD_HORIZONS_MINUTES:
    threshold_row = expansion_thresholds_gc.loc[expansion_thresholds_gc["horizon"].eq(h)].iloc[0]
    fit_mask = development_mask & atr_mask & forward_labels_gc[f"label_available_{h}"].to_numpy(dtype=bool)
    fit_values = forward_labels_gc.loc[fit_mask, f"future_range_{h}_atr"].to_numpy(dtype=float)
    recomputed = float(np.quantile(fit_values, EXPANSION_QUANTILE, method="linear"))
    threshold_checks[f"{h}_threshold_recomputed_from_Development"] = bool(np.isclose(recomputed, threshold_row["threshold"], rtol=0.0, atol=1e-12))
    threshold_checks[f"{h}_Development_count_exact"] = int(fit_mask.sum()) == int(threshold_row["development_observation_count_used"])
    threshold_checks[f"{h}_fitting_metadata_locked"] = threshold_row["fitting_partition"] == "Development" and threshold_row["normalization_denominator"] == "decision_atr_20m"

current_trusted_signature = {
    "object_id": id(research_bars), "shape": research_bars.shape,
    "columns": tuple(research_bars.columns), "dtypes": tuple(map(str, research_bars.dtypes)),
    "sample_hash": int(pd.util.hash_pandas_object(research_bars.iloc[TRUSTED_SAMPLE_INDEX], index=True).sum()),
}
production_validation_checks = pd.concat([
    validate_label_invariants(forward_labels_gc),
    pd.Series(source_boundary_checks, name="passed", dtype=bool),
    pd.Series(threshold_checks, name="passed", dtype=bool),
    pd.Series({
        "entry_price_matches_saved_entry_open": bool(np.allclose(forward_labels_gc["entry_price"].to_numpy(dtype=float), gc_path_source["open"].to_numpy(dtype=float)[entry_positions], rtol=0.0, atol=1e-12)),
        "tick_grid_validation_passed": int(tick_grid_validation["off_grid_prices"].sum()) == 0,
        "thresholds_use_no_Validation_or_Final_test": bool((expansion_thresholds_gc["fitting_partition"] == "Development").all() and synthetic_test_result.wasSuccessful()),
        "no_POI_columns_present": not any(re.search(r"(^|_)(poi|section6|section7)(_|$)", column.lower()) for column in forward_labels_gc.columns),
        "legacy_forward_columns_not_copied": not set(EXISTING_FORWARD_COLUMNS).intersection(forward_labels_gc.columns),
        "trusted_research_bars_not_mutated": current_trusted_signature == trusted_research_bars_signature,
    }, name="passed", dtype=bool),
])
display(production_validation_checks.to_frame())
assert production_validation_checks.all(), production_validation_checks[~production_validation_checks].index.tolist()

,passed
exactly_one_row_per_observation,True
observation_ids_ordered,True
observation_ids_match_section_3,True
entry_ids_match_section_3,True
entry_timestamps_match_section_3,True
...,...
tick_grid_validation_passed,True
thresholds_use_no_Validation_or_Final_test,True
no_POI_columns_present,True
legacy_forward_columns_not_copied,True


In [25]:
availability_report = build_label_availability_report(
    forward_labels_gc, horizons=FORWARD_HORIZONS_MINUTES
)
display(horizon_availability_counts)
display(availability_report)

audit_rng = np.random.default_rng(RANDOM_SEED)
audit_records = []
for h in FORWARD_HORIZONS_MINUTES:
    available = forward_labels_gc[f"label_available_{h}"].to_numpy(dtype=bool)
    for session in ("London", "New York"):
        candidates = np.flatnonzero(available & forward_labels_gc["entry_session"].eq(session).to_numpy(dtype=bool))
        row = int(audit_rng.choice(candidates))
        start = int(entry_positions[row])
        path = gc_path_source.iloc[start : start + h]
        path_high = float(path["high"].max())
        path_low = float(path["low"].min())
        path_closes = path["close"].to_numpy(dtype=float)
        entry_price = float(forward_labels_gc.loc[row, "entry_price"])
        path_log_returns = np.log(np.r_[path_closes[0] / entry_price, path_closes[1:] / path_closes[:-1]])
        reconstructed_rv = float(np.sqrt(np.square(path_log_returns).sum()) * 10_000.0)
        reconstructed_forward_ticks = int(np.rint((path_closes[-1] - entry_price) / GC_TICK_SIZE))
        reconstructed_mfe_ticks = int(np.rint((path_high - entry_price) / GC_TICK_SIZE))
        reconstructed_mae_ticks = int(np.rint((entry_price - path_low) / GC_TICK_SIZE))
        reconstructed_range_ticks = int(np.rint((path_high - path_low) / GC_TICK_SIZE))
        audit_passed = (
            reconstructed_forward_ticks == forward_labels_gc.loc[row, f"forward_return_{h}_ticks"]
            and reconstructed_mfe_ticks == forward_labels_gc.loc[row, f"mfe_long_{h}_ticks"]
            and reconstructed_mae_ticks == forward_labels_gc.loc[row, f"mae_long_{h}_ticks"]
            and reconstructed_range_ticks == forward_labels_gc.loc[row, f"future_range_{h}_ticks"]
            and np.isclose(reconstructed_rv, forward_labels_gc.loc[row, f"future_realized_volatility_{h}_bps"], rtol=0.0, atol=1e-7)
        )
        audit_records.append({
            "horizon": h, "session": session,
            "observation_id": int(forward_labels_gc.loc[row, "observation_id"]),
            "research_partition": str(forward_labels_gc.loc[row, "research_partition"]),
            "entry_timestamp_utc": forward_labels_gc.loc[row, "entry_timestamp_utc"],
            "entry_price": entry_price,
            "expected_final_path_timestamp_utc": path.iloc[-1]["ts_event_utc"],
            "economic_exit_timestamp_utc": forward_labels_gc.loc[row, f"exit_timestamp_utc_{h}"],
            "exit_price": float(path_closes[-1]), "maximum_high": path_high, "minimum_low": path_low,
            "forward_return_ticks": reconstructed_forward_ticks, "mfe_long_ticks": reconstructed_mfe_ticks,
            "mae_long_ticks": reconstructed_mae_ticks, "range_ticks": reconstructed_range_ticks,
            "realized_volatility_bps": reconstructed_rv, "availability_decision": bool(available[row]),
            "entry_symbol": path.iloc[0]["symbol"], "final_symbol": path.iloc[-1]["symbol"],
            "entry_instrument_id": int(path.iloc[0]["instrument_id"]),
            "final_instrument_id": int(path.iloc[-1]["instrument_id"]),
            "entry_segment_id": int(path.iloc[0]["continuous_segment_id"]),
            "final_segment_id": int(path.iloc[-1]["continuous_segment_id"]),
            "entry_trade_date_ny": path.iloc[0]["trade_date_ny"],
            "final_trade_date_ny": path.iloc[-1]["trade_date_ny"],
            "audit_passed": bool(audit_passed),
        })
manual_path_audits = pd.DataFrame(audit_records)
display(manual_path_audits)
assert len(manual_path_audits) == 2 * len(FORWARD_HORIZONS_MINUTES)
assert manual_path_audits["audit_passed"].all()

,horizon,available_observations,unavailable_observations,availability_rate
0,5,586468,62,0.999894
1,15,586296,234,0.999601
2,30,585958,572,0.999025
3,60,585033,1497,0.997448
4,120,582471,4059,0.993080
5,180,579166,7364,0.987445


,horizon,available_observations,unavailable_observations,availability_rate,unavailable_reason,reason_observations,session,research_partition
0,5,114822,12,0.999896,available,114822,London,Development
1,5,114822,12,0.999896,missing_or_nonconsecutive_minute,12,London,Development
2,5,60804,24,0.999605,available,60804,London,Final test
3,5,60804,24,0.999605,missing_or_nonconsecutive_minute,24,London,Final test
4,5,44266,10,0.999774,available,44266,London,Validation
5,5,44266,10,0.999774,missing_or_nonconsecutive_minute,10,London,Validation
6,5,191388,8,0.999958,available,191388,New York,Development
7,5,191388,8,0.999958,missing_or_nonconsecutive_minute,8,New York,Development
8,5,101388,8,0.999921,available,101388,New York,Final test
9,5,101388,8,0.999921,missing_or_nonconsecutive_minute,8,New York,Final test


,horizon,session,observation_id,research_partition,entry_timestamp_utc,entry_price,expected_final_path_timestamp_utc,economic_exit_timestamp_utc,exit_price,maximum_high,minimum_low,forward_return_ticks,mfe_long_ticks,mae_long_ticks,range_ticks,realized_volatility_bps,availability_decision,entry_symbol,final_symbol,entry_instrument_id,final_instrument_id,entry_segment_id,final_segment_id,entry_trade_date_ny,final_trade_date_ny,audit_passed
0,5,London,76371,Development,2022-01-12 08:51:00+00:00,"1,815.500000",2022-01-12 08:55:00+00:00,2022-01-12 08:56:00+00:00,"1,816.200000","1,816.300000","1,815.500000",7,8,0,8,2.523706,True,GCG2,GCG2,14725,14725,1173,1173,2022-01-12,2022-01-12,True
1,5,New York,562021,Final test,2026-03-05 16:31:00+00:00,"5,064.700000",2026-03-05 16:35:00+00:00,2026-03-05 16:36:00+00:00,"5,070.100000","5,076.600000","5,064.700000",54,119,0,119,18.345783,True,GCJ6,GCJ6,42000890,42000890,8178,8178,2026-03-05,2026-03-05,True
2,15,London,262129,Development,2023-08-18 07:55:00+00:00,"1,922.300000",2023-08-18 08:09:00+00:00,2023-08-18 08:10:00+00:00,"1,922.000000","1,922.600000","1,921.600000",-3,3,7,10,4.682588,True,GCZ3,GCZ3,7246,7246,3925,3925,2023-08-18,2023-08-18,True
3,15,New York,566616,Final test,2026-03-19 12:06:00+00:00,"4,637.200000",2026-03-19 12:20:00+00:00,2026-03-19 12:21:00+00:00,"4,620.100000","4,642.200000","4,611.300000",-171,50,259,309,40.369619,True,GCJ6,GCJ6,42000890,42000890,8207,8207,2026-03-19,2026-03-19,True
4,30,London,414388,Validation,2024-12-02 10:38:00+00:00,"2,659.300000",2024-12-02 11:07:00+00:00,2024-12-02 11:08:00+00:00,"2,660.600000","2,661.600000","2,656.800000",13,23,25,48,12.227751,True,GCG5,GCG5,1551,1551,6595,6595,2024-12-02,2024-12-02,True
5,30,New York,124579,Development,2022-06-15 12:23:00+00:00,"1,830.900000",2022-06-15 12:52:00+00:00,2022-06-15 12:53:00+00:00,"1,835.400000","1,838.500000","1,830.300000",45,76,6,82,29.997129,True,GCQ2,GCQ2,5568,5568,1932,1932,2022-06-15,2022-06-15,True
6,60,London,233400,Development,2023-05-18 09:06:00+00:00,"1,978.900000",2023-05-18 10:05:00+00:00,2023-05-18 10:06:00+00:00,"1,980.100000","1,980.700000","1,977.800000",12,18,11,29,12.498738,True,GCM3,GCM3,30052,30052,3145,3145,2023-05-18,2023-05-18,True
7,60,New York,82443,Development,2022-02-02 15:07:00+00:00,"1,804.100000",2022-02-02 16:06:00+00:00,2022-02-02 16:07:00+00:00,"1,809.300000","1,812.000000","1,803.300000",52,79,8,87,27.619462,True,GCJ2,GCJ2,1619,1619,1443,1443,2022-02-02,2022-02-02,True
8,120,London,470533,Final test,2025-05-23 09:33:00+00:00,"3,333.100000",2025-05-23 11:32:00+00:00,2025-05-23 11:33:00+00:00,"3,330.000000","3,333.700000","3,322.700000",-31,6,104,110,24.852703,True,GCM5,GCM5,287,287,7330,7330,2025-05-23,2025-05-23,True
9,120,New York,359770,Validation,2024-06-17 12:20:00+00:00,"2,333.700000",2024-06-17 14:19:00+00:00,2024-06-17 14:20:00+00:00,"2,336.900000","2,341.300000","2,331.300000",32,76,24,100,32.316672,True,GCQ4,GCQ4,2017,2017,5651,5651,2024-06-17,2024-06-17,True


## 4.7 Save Forward Label Table

The complete observation-preserving label table and frozen Development expansion thresholds are saved with PyArrow, Zstandard compression, and no index. An independent reload must preserve row and column order, Arrow dtypes and timezone metadata, categorical meanings, null semantics, identifiers, sampled content, and every label invariant before Section 4 can pass its final gate.

In [26]:
forward_labels_gc.to_parquet(
    FORWARD_LABELS_PATH, engine="pyarrow", compression="zstd", index=False
)
expansion_thresholds_gc.to_parquet(
    EXPANSION_THRESHOLDS_PATH, engine="pyarrow", compression="zstd", index=False
)

reloaded_forward_labels_gc = pd.read_parquet(
    FORWARD_LABELS_PATH, engine="pyarrow", dtype_backend="pyarrow"
)
reloaded_expansion_thresholds_gc = pd.read_parquet(
    EXPANSION_THRESHOLDS_PATH, engine="pyarrow", dtype_backend="pyarrow"
)
saved_forward_arrow = pq.read_table(FORWARD_LABELS_PATH)
saved_threshold_arrow = pq.read_table(EXPANSION_THRESHOLDS_PATH)
reloaded_forward_arrow = pa.Table.from_pandas(reloaded_forward_labels_gc, preserve_index=False)
reloaded_threshold_arrow = pa.Table.from_pandas(reloaded_expansion_thresholds_gc, preserve_index=False)
saved_forward_schema = pq.ParquetFile(FORWARD_LABELS_PATH).schema_arrow

def saved_dictionary_values(table: pa.Table, column: str) -> tuple[str, ...]:
    values = []
    for chunk in table[column].chunks:
        if pa.types.is_dictionary(chunk.type):
            for value in chunk.dictionary.to_pylist():
                if value not in values:
                    values.append(value)
    return tuple(values)

reload_invariant_checks = validate_label_invariants(reloaded_forward_labels_gc)
sample_positions = pa.array([0, len(forward_labels_gc) // 2, len(forward_labels_gc) - 1], type=pa.int64())
reason_dictionary_preserved = all(
    saved_dictionary_values(saved_forward_arrow, f"label_unavailable_reason_{h}") == LABEL_REASON_CATEGORIES
    for h in FORWARD_HORIZONS_MINUTES
)
timezone_columns = [
    "decision_timestamp_utc", "decision_timestamp_ny", "entry_timestamp_utc",
    "entry_timestamp_ny", "forced_exit_timestamp_ny",
] + [f"exit_timestamp_{zone}_{h}" for h in FORWARD_HORIZONS_MINUTES for zone in ("utc", "ny")]
timezone_metadata_preserved = all(
    pa.types.is_timestamp(saved_forward_schema.field(column).type)
    and saved_forward_schema.field(column).type.tz == ("UTC" if "_utc" in column else RESEARCH_TIMEZONE_NAME)
    for column in timezone_columns
)
reload_checks = pd.Series(
    {
        "row_count_preserved": len(reloaded_forward_labels_gc) == EXPECTED_SECTION_4_ROWS,
        "column_order_preserved": list(reloaded_forward_labels_gc.columns) == FORWARD_LABEL_COLUMN_ORDER,
        "Arrow_dtypes_preserved": reloaded_forward_arrow.schema.remove_metadata() == saved_forward_arrow.schema.remove_metadata(),
        "timezone_metadata_preserved": timezone_metadata_preserved,
        "reason_dictionary_meanings_preserved": reason_dictionary_preserved,
        "entry_session_meanings_preserved": saved_dictionary_values(saved_forward_arrow, "entry_session") == ("London", "New York"),
        "partition_meanings_preserved": saved_dictionary_values(saved_forward_arrow, "research_partition") == ("Development", "Validation", "Final test"),
        "null_counts_preserved": bool(reloaded_forward_labels_gc.isna().sum().eq(forward_labels_gc.isna().sum()).all()),
        "observation_ids_unique_and_ordered": reloaded_forward_labels_gc["observation_id"].is_unique and reloaded_forward_labels_gc["observation_id"].is_monotonic_increasing,
        "sampled_content_preserved": reloaded_forward_arrow.take(sample_positions).equals(saved_forward_arrow.take(sample_positions)),
        "threshold_rows_preserved": len(reloaded_expansion_thresholds_gc) == len(FORWARD_HORIZONS_MINUTES),
        "threshold_schema_preserved": reloaded_threshold_arrow.schema.remove_metadata() == saved_threshold_arrow.schema.remove_metadata(),
        "all_reloaded_label_invariants_pass": bool(reload_invariant_checks.all()),
    },
    name="passed",
    dtype=bool,
)
display(reload_checks.to_frame())
assert reload_checks.all(), reload_checks[~reload_checks].index.tolist()

forward_labels_gc = reloaded_forward_labels_gc
expansion_thresholds_gc = reloaded_expansion_thresholds_gc
section_4_ready = bool(
    synthetic_test_result.wasSuccessful()
    and production_validation_checks.all()
    and manual_path_audits["audit_passed"].all()
    and reload_checks.all()
)
print(f"Saved output: {FORWARD_LABELS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Forward label shape: {forward_labels_gc.shape}")
print(f"Forward label file size: {FORWARD_LABELS_PATH.stat().st_size / (1024**2):,.2f} MiB")
print(f"Saved thresholds: {EXPANSION_THRESHOLDS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Availability counts: {dict(zip(horizon_availability_counts['horizon'], horizon_availability_counts['available_observations']))}")
print("Expansion thresholds:")
display(expansion_thresholds_gc)
print("Section 5 remains unstarted.")
print("SECTION 4 STATUS: READY" if section_4_ready else "SECTION 4 STATUS: NOT READY")
assert section_4_ready

,passed
row_count_preserved,True
column_order_preserved,True
Arrow_dtypes_preserved,True
timezone_metadata_preserved,True
reason_dictionary_meanings_preserved,True
entry_session_meanings_preserved,True
partition_meanings_preserved,True
null_counts_preserved,True
observation_ids_unique_and_ordered,True
sampled_content_preserved,True


Saved output: data\processed\statistical_research\forward_labels_gc.parquet
Forward label shape: (586530, 169)
Forward label file size: 270.23 MiB
Saved thresholds: data\processed\statistical_research\forward_label_expansion_thresholds_gc.parquet
Availability counts: {5: 586468, 15: 586296, 30: 585958, 60: 585033, 120: 582471, 180: 579166}
Expansion thresholds:


,horizon,quantile,threshold,development_observation_count_used,fitting_partition,normalization_denominator
0,5,0.800000,3.111111,306161,Development,decision_atr_20m
1,15,0.800000,5.557772,306108,Development,decision_atr_20m
2,30,0.800000,8.000000,305979,Development,decision_atr_20m
3,60,0.800000,11.797753,305603,Development,decision_atr_20m
4,120,0.800000,17.413793,304370,Development,decision_atr_20m
5,180,0.800000,22.409639,302774,Development,decision_atr_20m


Section 5 remains unstarted.
SECTION 4 STATUS: READY


# 5.0 Establish Baseline Behaviour

Section 5 asks what an eligible GC observation naturally produces before any predictive feature is used. It is a descriptive event study, not a strategy, trade simulation, feature screen, holding-period choice, or profitability claim. The metric definitions, horizons, fixed hurdle grid, grouping rules, and date-block uncertainty method below are frozen before this one-time descriptive exposure to Final-test outcomes. No Section 6 work is started here.


## 5.1 Baseline Scope, Metrics, and Comparison Rules

The population is the complete 586,530-row GC eligible-observation frame validated in Sections 3 and 4. Outcomes use the exact 5, 15, 30, 60, 120, and 180 minute fixed-horizon labels. A horizon contributes only when its `label_available_<horizon>` flag is true; a fixed path is never shortened to the forced exit. Raw tick, basis-point, ATR-normalized, excursion, range, realized-volatility, time-to-extreme, direction, and expansion outcomes retain their Section 4 definitions.

Minute observations and horizons overlap, so rows are serially dependent and are not independent trials. Every table reports both observation count and New York trading-date coverage. Headline means, directional rates, and London-minus-New-York differences use a reproducible New York trading-date block bootstrap on daily aggregates. These intervals describe uncertainty; they do not prove an edge.

London and New York are displayed independently before combined interpretations because their sample sizes, movement regimes, and long-horizon availability can differ. Means are accompanied by medians, robust centers, dispersion, central and tail quantiles, and sign rates because GC outcomes are fat-tailed. MFE is not realized profit, MAE is not realized loss, and future range is not a tradable return. A future feature must beat the matching baseline under the same horizon, session, direction, partition, availability, outcome, hurdle, and date coverage.


In [27]:
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.statistical_research.baselines import (
    BASELINE_RANDOM_SEED,
    BOOTSTRAP_REPLICATES,
    COST_THRESHOLDS_TICKS,
    FORWARD_HORIZONS_MINUTES,
    build_baseline_outputs,
    save_baseline_outputs,
)

assert section_4_ready, "Section 5 may not use labels until Section 4 is READY."
assert forward_labels_gc.shape == (586_530, 169)

baseline_configuration = pd.Series(
    {
        "instrument": "GC",
        "population": "all Section 4 eligible observations",
        "horizons_minutes": FORWARD_HORIZONS_MINUTES,
        "time_bin_minutes": 15,
        "cost_threshold_ticks": COST_THRESHOLDS_TICKS,
        "bootstrap_unit": "New York trading date / daily aggregate",
        "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        "random_seed": BASELINE_RANDOM_SEED,
        "final_test_governance": "one-time predeclared descriptive exposure",
    },
    name="value",
)
display(baseline_configuration.to_frame())


,value
instrument,GC
population,all Section 4 eligible observations
horizons_minutes,"(5, 15, 30, 60, 120, 180)"
time_bin_minutes,15
cost_threshold_ticks,"(0, 1, 2, 3, 4, 5, 10)"
bootstrap_unit,New York trading date / daily aggregate
bootstrap_replicates,2000
random_seed,20260714
final_test_governance,one-time predeclared descriptive exposure


## 5.2 Unconditional Forward Outcome Distributions

This subsection establishes statistical center, typical movement, tail movement, and directional balance before conditioning. The engine processes compact outcome families and retains raw values for calculation; chart limits never alter the underlying summaries.


In [28]:
baseline_result = build_baseline_outputs(forward_labels_gc, section_4_ready=section_4_ready)
assert baseline_result.validation.loc[baseline_result.validation["critical"], "passed"].all()

unconditional_returns = baseline_result.unconditional.loc[
    baseline_result.unconditional["outcome_name"].isin(
        ["forward_return_ticks", "forward_return_bps", "forward_return_atr", "direction_label"]
    ),
    [
        "outcome_name", "horizon_minutes", "observation_count", "trading_date_count",
        "available_observation_count", "availability_rate", "mean", "median",
        "standard_deviation", "mean_absolute_value", "p01", "p05", "p25", "p75",
        "p95", "p99", "positive_rate", "negative_rate", "zero_rate", "trimmed_mean",
    ],
]
display(unconditional_returns)

return_ticks = unconditional_returns.loc[unconditional_returns["outcome_name"].eq("forward_return_ticks")]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for quantile in ("p05", "p25", "median", "p75", "p95"):
    axes[0].plot(return_ticks["horizon_minutes"], return_ticks[quantile], marker="o", label=quantile)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(title="Raw forward-return quantiles", xlabel="Horizon (minutes)", ylabel="Ticks")
axes[0].legend(ncol=3, fontsize=8)

headline_ci = baseline_result.headline_uncertainty.query(
    "analysis_type == 'headline_date_block_bootstrap' and session == 'Combined' and metric_name == 'mean_return'"
).sort_values("horizon_minutes")
axes[1].errorbar(
    headline_ci["horizon_minutes"], headline_ci["metric_value"],
    yerr=[headline_ci["metric_value"] - headline_ci["confidence_interval_lower"], headline_ci["confidence_interval_upper"] - headline_ci["metric_value"]],
    marker="o", capsize=3,
)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="Daily-weighted mean with date-block interval", xlabel="Horizon (minutes)", ylabel="Ticks")
fig.tight_layout()
plt.show()


,outcome_name,horizon_minutes,observation_count,trading_date_count,available_observation_count,availability_rate,mean,median,standard_deviation,mean_absolute_value,p01,p05,p25,p75,p95,p99,positive_rate,negative_rate,zero_rate,trimmed_mean
0,forward_return_ticks,5,586530,1222,586468,0.999894,0.014173,0.000000,25.753310,14.191543,-74.000000,-31.000000,-8.000000,8.000000,31.000000,71.000000,0.485657,0.477421,0.036923,0.099904
1,forward_return_bps,5,586530,1222,586468,0.999894,0.012030,0.000000,7.997285,5.259660,-22.385313,-11.493517,-3.537858,3.660935,11.422769,21.390824,0.485657,0.477421,0.036923,0.030888
2,forward_return_atr,5,586530,1222,586320,0.999642,0.004860,0.000000,1.644629,1.215314,-4.150943,-2.541436,-0.945946,0.967742,2.518519,4.033613,0.485672,0.477427,0.036901,0.006183
3,direction_label,5,586530,1222,586468,0.999894,0.008236,0.000000,0.981331,0.963077,-1.000000,-1.000000,-1.000000,1.000000,1.000000,1.000000,0.485657,0.477421,0.036923,0.008404
24,forward_return_ticks,15,586530,1222,586296,0.999601,-0.035670,0.000000,43.707777,24.453619,-132.000000,-55.000000,-14.000000,15.000000,54.000000,120.000000,0.496817,0.482065,0.021117,0.221793
25,forward_return_bps,15,586530,1222,586296,0.999601,0.012260,0.000000,13.706500,9.088028,-39.716025,-20.082609,-6.024096,6.363191,19.738751,36.638579,0.496817,0.482065,0.021117,0.071845
26,forward_return_atr,15,586530,1222,586154,0.999359,0.011006,0.000000,2.921326,2.123866,-7.532468,-4.485981,-1.612903,1.666667,4.403670,7.252747,0.496796,0.482085,0.021119,0.015865
27,direction_label,15,586530,1222,586296,0.999601,0.014752,0.000000,0.989276,0.978883,-1.000000,-1.000000,-1.000000,1.000000,1.000000,1.000000,0.496817,0.482065,0.021117,0.015053
48,forward_return_ticks,30,586530,1222,585958,0.999025,-0.144178,1.000000,60.394770,34.491711,-189.000000,-78.000000,-19.000000,21.000000,77.000000,165.000000,0.500683,0.484188,0.015129,0.277765
49,forward_return_bps,30,586530,1222,585958,0.999025,0.003234,0.263002,19.205693,12.860469,-57.270269,-28.854841,-8.467042,9.077808,27.883920,51.389304,0.500683,0.484188,0.015129,0.093544


C:\Users\abond\AppData\Local\Temp\ipykernel_101100\648958941.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The unconditional center remains small relative to typical and tail movement. This is a benchmark, not evidence that conditioning cannot work. Raw event-weighted means and daily-weighted bootstrap centers answer slightly different weighting questions and are labeled accordingly.


## 5.3 Baseline Excursion, Range, and Volatility Behaviour

Path opportunity, path risk, total high-low range, realized path volatility, and the timing of first extremes are summarized independently of any stop, target, or holding-period rule. Horizon fractions make time-to-extreme comparable without changing the one-based Section 4 timing labels.


In [29]:
movement_names = [
    "mfe_long_ticks", "mae_long_ticks", "mfe_short_ticks", "mae_short_ticks",
    "future_range_ticks", "future_range_atr", "future_realized_volatility_bps",
    "time_to_mfe_long_minutes", "time_to_mae_long_minutes",
    "time_to_mfe_long_fraction", "time_to_mae_long_fraction",
]
movement_summary = baseline_result.unconditional.loc[
    baseline_result.unconditional["outcome_name"].isin(movement_names),
    ["outcome_name", "horizon_minutes", "available_observation_count", "trading_date_count", "mean", "median", "p05", "p25", "p75", "p95", "p99"],
]
display(movement_summary)

range_profile = movement_summary.loc[movement_summary["outcome_name"].eq("future_range_ticks")]
vol_profile = movement_summary.loc[movement_summary["outcome_name"].eq("future_realized_volatility_bps")]
fig, ax1 = plt.subplots(figsize=(8.5, 4.2))
ax1.plot(range_profile["horizon_minutes"], range_profile["mean"], marker="o", color="tab:blue", label="Mean future range")
ax1.set(xlabel="Horizon (minutes)", ylabel="Future range (ticks)")
ax2 = ax1.twinx()
ax2.plot(vol_profile["horizon_minutes"], vol_profile["mean"], marker="s", color="tab:orange", label="Mean realized volatility")
ax2.set_ylabel("Realized volatility (bps)")
ax1.set_title("Non-linear movement growth across horizons")
fig.tight_layout()
plt.show()


,outcome_name,horizon_minutes,available_observation_count,trading_date_count,mean,median,p05,p25,p75,p95,p99
4,mfe_long_ticks,5,586468,1222,14.108898,8.000000,0.000000,3.000000,17.000000,46.000000,96.000000
5,mae_long_ticks,5,586468,1222,14.601144,8.000000,0.000000,3.000000,17.000000,49.000000,106.000000
6,mfe_short_ticks,5,586468,1222,14.601144,8.000000,0.000000,3.000000,17.000000,49.000000,106.000000
7,mae_short_ticks,5,586468,1222,14.108898,8.000000,0.000000,3.000000,17.000000,46.000000,96.000000
12,time_to_mfe_long_minutes,5,586468,1222,2.813173,3.000000,1.000000,1.000000,4.000000,5.000000,5.000000
13,time_to_mae_long_minutes,5,586468,1222,2.791593,3.000000,1.000000,1.000000,4.000000,5.000000,5.000000
16,time_to_mfe_long_fraction,5,586468,1222,0.562635,0.600000,0.200000,0.200000,0.800000,1.000000,1.000000
17,time_to_mae_long_fraction,5,586468,1222,0.558319,0.600000,0.200000,0.200000,0.800000,1.000000,1.000000
20,future_range_ticks,5,586468,1222,28.710042,20.000000,7.000000,12.000000,33.000000,80.000000,156.000000
21,future_range_atr,5,586320,1222,2.450556,2.191781,1.190131,1.690141,2.884615,4.493392,6.666667


C:\Users\abond\AppData\Local\Temp\ipykernel_101100\1865491346.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Excursions and range expand with horizon, but the compact tables—not visual scaling—define the benchmark. Long/short excursion identities come from the same path and are paired quantities. No horizon, stop, or target is selected here.


## 5.4 Forward Outcomes by Session

London and New York are evaluated with identical definitions. Absolute results appear before their difference; counts and availability remain visible so the larger New York sample cannot silently dominate a combined headline.


In [30]:
session_key = baseline_result.session.loc[
    baseline_result.session["outcome_name"].isin(
        ["forward_return_ticks", "mfe_long_ticks", "mae_long_ticks", "future_range_ticks", "future_realized_volatility_bps"]
    ),
    ["entry_session", "outcome_name", "horizon_minutes", "observation_count", "trading_date_count", "available_observation_count", "availability_rate", "mean", "median", "standard_deviation", "mean_absolute_value", "p05", "p95", "positive_rate"],
].sort_values(["entry_session", "outcome_name", "horizon_minutes"])
display(session_key)

session_difference_ci = baseline_result.headline_uncertainty.query(
    "analysis_type == 'session_difference_date_block_bootstrap'"
).sort_values(["metric_name", "horizon_minutes"])
display(session_difference_ci)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for session in ("London", "New York"):
    data = session_key.query("entry_session == @session and outcome_name == 'future_range_ticks'")
    axes[0].plot(data["horizon_minutes"], data["mean"], marker="o", label=session)
    availability = session_key.query("entry_session == @session and outcome_name == 'forward_return_ticks'")
    axes[1].plot(availability["horizon_minutes"], availability["availability_rate"], marker="o", label=session)
axes[0].set(title="Mean future range by session", xlabel="Horizon (minutes)", ylabel="Ticks")
axes[1].set(title="Availability by session", xlabel="Horizon (minutes)", ylabel="Rate", ylim=(0.95, 1.001))
for ax in axes: ax.legend()
fig.tight_layout(); plt.show()


,entry_session,outcome_name,horizon_minutes,observation_count,trading_date_count,available_observation_count,availability_rate,mean,median,standard_deviation,mean_absolute_value,p05,p95,positive_rate
0,London,forward_return_ticks,5,219938,1222,219892,0.999791,0.074264,0.000000,19.184944,10.623370,-23.000000,23.000000,0.479831
20,London,forward_return_ticks,15,219938,1222,219760,0.999191,0.112614,0.000000,32.414241,18.026010,-38.000000,39.000000,0.491454
40,London,forward_return_ticks,30,219938,1222,219513,0.998068,0.164332,0.000000,44.323728,25.034431,-54.000000,55.000000,0.496344
60,London,forward_return_ticks,60,219938,1222,218904,0.995299,-0.025376,1.000000,62.420224,34.866375,-75.000000,74.000000,0.507058
80,London,forward_return_ticks,120,219938,1222,217856,0.990534,0.508804,2.000000,86.712165,47.631279,-102.000000,98.000000,0.516304
100,London,forward_return_ticks,180,219938,1222,217079,0.987001,0.862958,3.000000,113.544296,61.666149,-130.000000,128.000000,0.517862
14,London,future_range_ticks,5,219938,1222,219892,0.999791,21.520137,15.000000,24.144359,21.520137,6.000000,59.000000,1.000000
34,London,future_range_ticks,15,219938,1222,219760,0.999191,37.743011,26.000000,41.412797,37.743011,11.000000,103.000000,1.000000
54,London,future_range_ticks,30,219938,1222,219513,0.998068,53.258121,37.000000,57.644541,53.258121,16.000000,145.000000,1.000000
74,London,future_range_ticks,60,219938,1222,218904,0.995299,74.682738,51.000000,81.497484,74.682738,22.000000,204.000000,1.000000


,analysis_type,horizon_minutes,session,metric_name,metric_value,confidence_interval_lower,confidence_interval_upper,trading_date_count,bootstrap_replicates,random_seed
6,session_difference_date_block_bootstrap,5,London minus New York,mean_return,0.096218,-0.163668,0.350049,1222,2000,20260714
14,session_difference_date_block_bootstrap,15,London minus New York,mean_return,0.237798,-0.507849,0.989386,1222,2000,20260714
22,session_difference_date_block_bootstrap,30,London minus New York,mean_return,0.498144,-1.013420,2.031878,1222,2000,20260714
30,session_difference_date_block_bootstrap,60,London minus New York,mean_return,0.856687,-2.221609,3.781494,1222,2000,20260714
38,session_difference_date_block_bootstrap,120,London minus New York,mean_return,2.309685,-3.292589,8.087900,1221,2000,20260714
46,session_difference_date_block_bootstrap,180,London minus New York,mean_return,2.853125,-4.580652,10.125108,1219,2000,20260714
7,session_difference_date_block_bootstrap,5,London minus New York,positive_rate,-0.009321,-0.014060,-0.004568,1222,2000,20260714
15,session_difference_date_block_bootstrap,15,London minus New York,positive_rate,-0.008556,-0.016449,-0.000211,1222,2000,20260714
23,session_difference_date_block_bootstrap,30,London minus New York,positive_rate,-0.006884,-0.018437,0.004815,1222,2000,20260714
31,session_difference_date_block_bootstrap,60,London minus New York,positive_rate,0.001067,-0.015236,0.017492,1222,2000,20260714


C:\Users\abond\AppData\Local\Temp\ipykernel_101100\3951732107.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


New York produces materially larger movement magnitudes than London, while signed-return centers remain economically small relative to dispersion. A detectable session difference is not automatically a trading edge.


## 5.5 Forward Outcomes by Time of Day and Day of Week

All grouping uses timezone-aware `entry_timestamp_ny`. Stable 15-minute clock bins and 15-minute session-relative bins expose both within-session evolution and declining long-horizon availability. Weekdays use New York entry dates and are also saved by year to diagnose instability and partial-year dependence.


In [31]:
tod_return = baseline_result.time_of_day.loc[
    baseline_result.time_of_day["outcome_name"].eq("forward_return_ticks"),
    ["entry_session", "time_of_day_bin", "minutes_since_session_open_bin", "horizon_minutes", "observation_count", "trading_date_count", "availability_rate", "mean", "median", "positive_rate"],
]
display(tod_return.groupby(["entry_session", "horizon_minutes"], observed=True).tail(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.0))
for ax, session in zip(axes, ("London", "New York"), strict=True):
    heat = baseline_result.time_of_day.query("entry_session == @session and outcome_name == 'future_range_ticks'").pivot(index="time_of_day_bin", columns="horizon_minutes", values="mean")
    image = ax.imshow(heat.to_numpy(), aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(heat.columns)), labels=heat.columns)
    step = max(1, len(heat) // 10)
    ax.set_yticks(range(0, len(heat), step), labels=heat.index[::step])
    ax.set(title=f"{session}: mean future range", xlabel="Horizon (minutes)", ylabel="Entry time bin")
    fig.colorbar(image, ax=ax, label="Ticks")
fig.tight_layout(); plt.show()

weekday_overall = baseline_result.weekday.query("analysis_type == 'weekday' and outcome_name == 'forward_return_ticks'")
weekday_by_year = baseline_result.weekday.query("analysis_type == 'weekday_by_year' and outcome_name == 'forward_return_ticks'")
display(weekday_overall[["day_of_week", "entry_session", "horizon_minutes", "observation_count", "trading_date_count", "availability_rate", "mean", "median", "positive_rate"]])
display(weekday_by_year.groupby(["year", "day_of_week", "entry_session"], observed=True)["trading_date_count"].max().reset_index())


,entry_session,time_of_day_bin,minutes_since_session_open_bin,horizon_minutes,observation_count,trading_date_count,availability_rate,mean,median,positive_rate
10,London,05:30,150-164,5,18326,1222,0.999563,0.099465,0.000000,0.491047
11,London,05:45,165-179,5,18330,1222,0.999891,0.245308,0.000000,0.485159
30,New York,11:30,270-284,5,18324,1222,0.999345,-0.170216,0.000000,0.484709
31,New York,11:45,285-299,5,18328,1222,0.999782,0.058448,0.000000,0.489358
330,London,05:30,150-164,15,18326,1222,0.999072,-0.038833,1.000000,0.506964
331,London,05:45,165-179,15,18330,1222,0.998691,0.476128,0.000000,0.497433
350,New York,11:30,270-284,15,18324,1222,0.997708,-0.183897,1.000000,0.508369
351,New York,11:45,285-299,15,18328,1222,0.999782,-0.320181,1.000000,0.501528
650,London,05:30,150-164,30,18326,1222,0.997763,0.436970,1.000000,0.509707
651,London,05:45,165-179,30,18330,1222,0.994872,0.677616,1.000000,0.509871


C:\Users\abond\AppData\Local\Temp\ipykernel_101100\1800355333.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


,day_of_week,entry_session,horizon_minutes,observation_count,trading_date_count,availability_rate,mean,median,positive_rate
0,Friday,London,5,42838,238,0.999860,0.096470,0.000000,0.485922
1,Friday,New York,5,71400,238,1.000000,0.282241,0.000000,0.491387
2,Monday,London,5,44456,247,0.999820,0.244015,0.000000,0.481169
3,Monday,New York,5,74094,247,0.999838,-0.095367,0.000000,0.485206
4,Thursday,London,5,43914,244,0.999727,0.208123,0.000000,0.479318
5,Thursday,New York,5,73198,244,0.999945,-0.205222,0.000000,0.485928
6,Tuesday,London,5,44994,250,0.999733,-0.108332,0.000000,0.475190
7,Tuesday,New York,5,75000,250,1.000000,-0.021187,0.000000,0.493453
8,Wednesday,London,5,43736,243,0.999817,-0.066593,0.000000,0.477795
9,Wednesday,New York,5,72900,243,1.000000,-0.061660,0.000000,0.489781


,year,day_of_week,entry_session,trading_date_count
0,"2,021.000000",Friday,London,29
1,"2,021.000000",Friday,New York,29
2,"2,021.000000",Monday,London,31
3,"2,021.000000",Monday,New York,31
4,"2,021.000000",Thursday,London,30
5,"2,021.000000",Thursday,New York,30
6,"2,021.000000",Tuesday,London,31
7,"2,021.000000",Tuesday,New York,31
8,"2,021.000000",Wednesday,London,31
9,"2,021.000000",Wednesday,New York,31


Near-cutoff bins retain the same observation population but can lose complete long paths through data gaps, which is visible in the availability field. Weekday differences are treated as unstable unless they persist across years and adequate date counts; Section 5 does not promote a weekday filter.


## 5.6 Directional Symmetry and Tail Asymmetry

Short signed return is exactly the negative of long signed return, so the two framings are paired transformations rather than independent samples. Excursion identities, positive rates, upside/downside tails, tail magnitudes, and MFE/MAE ratios are compared by horizon, session, year, and frozen partition.


In [32]:
symmetry_headline = baseline_result.directional_symmetry.query("grouping in ['full', 'session']")
display(symmetry_headline[[
    "grouping", "group_value", "horizon_minutes", "available_observation_count", "trading_date_count",
    "mean_long_return_ticks", "mean_short_return_ticks", "long_positive_rate", "short_positive_rate",
    "mean_long_mfe_ticks", "mean_short_mfe_ticks", "p95_ticks", "absolute_p05_ticks",
    "p99_ticks", "absolute_p01_ticks", "p95_tail_ratio", "p99_tail_ratio",
    "tail_magnitude_ratio", "long_mfe_to_mae_ratio", "short_mfe_to_mae_ratio",
]])


,grouping,group_value,horizon_minutes,available_observation_count,trading_date_count,mean_long_return_ticks,mean_short_return_ticks,long_positive_rate,short_positive_rate,mean_long_mfe_ticks,mean_short_mfe_ticks,p95_ticks,absolute_p05_ticks,p99_ticks,absolute_p01_ticks,p95_tail_ratio,p99_tail_ratio,tail_magnitude_ratio,long_mfe_to_mae_ratio,short_mfe_to_mae_ratio
0,full,Combined,5,586468,1222,0.014173,-0.014173,0.485657,0.477421,14.108898,14.601144,31.000000,31.000000,71.000000,74.000000,1.000000,0.959459,0.960187,0.966287,1.034889
1,session,London,5,219892,1222,0.074264,-0.074264,0.479831,0.473928,10.667578,10.852559,23.000000,23.000000,53.000000,54.000000,1.000000,0.981481,0.964629,0.982955,1.017341
2,session,New York,5,366576,1222,-0.021873,0.021873,0.489151,0.479516,16.173186,16.849747,36.000000,36.000000,80.000000,84.000000,1.000000,0.952381,0.958485,0.959847,1.041832
12,full,Combined,15,586296,1222,-0.035670,0.035670,0.496817,0.482065,24.620736,26.090894,54.000000,55.000000,120.000000,132.000000,0.981818,0.909091,0.913688,0.943652,1.059712
13,session,London,15,219760,1222,0.112614,-0.112614,0.491454,0.481971,18.549509,19.193502,39.000000,38.000000,87.000000,93.000000,1.026316,0.935484,0.972206,0.966447,1.034718
14,session,New York,15,366536,1222,-0.124574,0.124574,0.500033,0.482122,28.260796,30.226289,62.000000,64.000000,135.000000,150.000000,0.968750,0.900000,0.901359,0.934974,1.069548
24,full,Combined,30,585958,1222,-0.144178,0.144178,0.500683,0.484188,34.834131,37.539327,77.000000,78.000000,165.000000,189.000000,0.987179,0.873016,0.902008,0.927937,1.077659
25,session,London,30,219513,1222,0.164332,-0.164332,0.496344,0.484236,25.995385,27.262736,55.000000,54.000000,118.000000,130.000000,1.018519,0.907692,0.961128,0.953513,1.048753
26,session,New York,30,366445,1222,-0.328985,0.328985,0.503282,0.484160,40.128841,43.695354,88.000000,92.000000,185.000000,212.000000,0.956522,0.872642,0.879376,0.918378,1.088877
36,full,Combined,60,585033,1222,-0.536242,0.536242,0.506457,0.483002,49.217109,54.006220,108.000000,111.000000,227.000000,263.000000,0.972973,0.863118,0.877519,0.911323,1.097306


Central p95/p05 tails are broadly balanced, while extreme p99/p01 ratios increasingly favor larger downside magnitude at longer horizons. This modest unconditional asymmetry supports preserving direction-aware diagnostics, not an unconditional short thesis.


## 5.7 Year and Research-Partition Stability

New York entry years and frozen research partitions are reported transparently. Partial 2021 and partial 2026 are labeled. Development and Validation drive interpretation; Final test is exposed once under the predeclared metric contract and may not be used to revise it.


In [33]:
display(baseline_result.year_coverage)

stability_key = baseline_result.year_partition_stability.loc[
    baseline_result.year_partition_stability["outcome_name"].isin(
        ["forward_return_ticks", "mfe_long_ticks", "mae_long_ticks", "future_range_ticks", "future_realized_volatility_bps"]
    ),
    ["analysis_type", "year", "research_partition", "entry_session", "outcome_name", "horizon_minutes", "observation_count", "trading_date_count", "availability_rate", "mean", "median", "positive_rate", "p05", "p95"],
]
display(stability_key)


,year,first_entry_date,last_entry_date,trading_date_count,coverage_status
0,2021,2021-05-24,2021-12-31,152,partial from dataset start
1,2022,2022-01-03,2022-12-30,244,complete observed year
2,2023,2023-01-03,2023-12-29,242,complete observed year
3,2024,2024-01-02,2024-12-31,246,complete observed year
4,2025,2025-01-02,2025-12-31,245,complete observed year
5,2026,2026-01-02,2026-05-22,93,partial through 2026-05-22


,analysis_type,year,research_partition,entry_session,outcome_name,horizon_minutes,observation_count,trading_date_count,availability_rate,mean,median,positive_rate,p05,p95
0,year_stability,"2,021.000000",NaN,London,forward_return_ticks,5,27360,152,1.000000,-0.062865,0.000000,0.463231,-12.000000,12.000000
1,year_stability,"2,021.000000",NaN,New York,forward_return_ticks,5,45600,152,1.000000,-0.022807,0.000000,0.484737,-21.000000,21.000000
2,year_stability,"2,022.000000",NaN,London,forward_return_ticks,5,43918,244,0.999909,0.043881,0.000000,0.477001,-15.000000,16.000000
3,year_stability,"2,022.000000",NaN,New York,forward_return_ticks,5,73196,244,0.999891,0.114090,0.000000,0.485312,-23.000000,24.000000
4,year_stability,"2,023.000000",NaN,London,forward_return_ticks,5,43556,242,0.999816,0.016442,0.000000,0.469115,-12.000000,13.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1075,partition_stability,NaN,Development,New York,future_realized_volatility_bps,180,191396,638,0.986760,42.146092,39.814954,1.000000,19.909153,73.591335
1076,partition_stability,NaN,Final test,London,future_realized_volatility_bps,180,60828,338,0.978661,41.665655,35.088212,1.000000,21.364214,79.277966
1077,partition_stability,NaN,Final test,New York,future_realized_volatility_bps,180,101396,338,0.988323,57.844695,51.453197,1.000000,24.822774,116.865598
1078,partition_stability,NaN,Validation,London,future_realized_volatility_bps,180,44276,246,0.985071,26.937412,25.105362,1.000000,16.548272,41.868187


Year and partition tables expose sign consistency, dispersion, session dependence, and exceptional-year sensitivity directly. No opaque composite stability score is introduced, and partial years are not allowed to masquerade as full-year replications.


## 5.8 Economic Significance and Transaction-Cost Thresholds

No approved full friction model exists in this branch, so Section 5 uses the fixed predeclared grid 0, 1, 2, 3, 4, 5, and 10 ticks. Exit-to-exit return hurdles, MFE opportunity, MAE risk, and total high-low range hurdles are distinct and must not be interpreted as PnL.


In [34]:
cost_display = baseline_result.cost_thresholds.loc[
    baseline_result.cost_thresholds["session"].isin(["London", "New York", "Combined"]),
    [
        "horizon_minutes", "session", "cost_threshold_ticks", "available_observation_count", "trading_date_count", "availability_rate",
        "probability_return_above_positive_threshold", "probability_return_below_negative_threshold",
        "probability_absolute_return_above_threshold", "mean_excess_movement_beyond_threshold_ticks",
        "median_excess_movement_beyond_threshold_ticks", "mfe_long_exceedance_rate",
        "mae_long_exceedance_rate", "future_range_exceedance_rate",
    ],
]
display(cost_display)


,horizon_minutes,session,cost_threshold_ticks,available_observation_count,trading_date_count,availability_rate,probability_return_above_positive_threshold,probability_return_below_negative_threshold,probability_absolute_return_above_threshold,mean_excess_movement_beyond_threshold_ticks,median_excess_movement_beyond_threshold_ticks,mfe_long_exceedance_rate,mae_long_exceedance_rate,future_range_exceedance_rate
0,5,London,0,219892,1222,0.999791,0.479831,0.473928,0.953759,11.138420,7.000000,0.920215,0.919706,1.000000
1,5,London,1,219892,1222,0.999791,0.435391,0.429024,0.864415,11.186302,7.000000,0.838775,0.838030,0.999991
2,5,London,2,219892,1222,0.999791,0.392429,0.385648,0.778077,11.316606,7.000000,0.760351,0.758531,0.999495
3,5,London,3,219892,1222,0.999791,0.351536,0.345652,0.697188,11.513568,7.000000,0.685473,0.682899,0.995589
4,5,London,4,219892,1222,0.999791,0.313745,0.308765,0.622510,11.774796,7.000000,0.615825,0.612851,0.982955
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,180,Combined,2,579166,1222,0.987445,0.498577,0.471138,0.969715,83.186393,49.000000,0.969995,0.967215,1.000000
122,180,Combined,3,579166,1222,0.987445,0.492677,0.465062,0.957739,83.214075,49.000000,0.959882,0.956140,1.000000
123,180,Combined,4,579166,1222,0.987445,0.486676,0.458962,0.945637,83.266217,49.000000,0.949374,0.944904,1.000000
124,180,Combined,5,579166,1222,0.987445,0.480577,0.452862,0.933439,83.341307,49.000000,0.938688,0.933643,1.000000


Movement frequently exceeds small tick hurdles, especially intrahorizon range and excursion hurdles, but no execution rule is supplied for harvesting those paths. These are friction-sensitivity benchmarks, not trade-cost-adjusted returns.


## 5.9 Baseline Validation and Save Outputs

Critical input, availability, boundary, representation, classification, aggregation, quantile, sampled-audit, and output-schema checks must all pass before Section 5 is saved or marked ready. Failed checks remain visible and block the final gate.


In [35]:
entry_minutes = pd.to_datetime(forward_labels_gc["entry_timestamp_ny"]).dt.hour * 60 + pd.to_datetime(forward_labels_gc["entry_timestamp_ny"]).dt.minute
audit_masks = {
    "London early": forward_labels_gc["entry_session"].astype(str).eq("London") & entry_minutes.lt(195),
    "London near close": forward_labels_gc["entry_session"].astype(str).eq("London") & entry_minutes.ge(345),
    "New York early": forward_labels_gc["entry_session"].astype(str).eq("New York") & entry_minutes.lt(435),
    "New York near noon": forward_labels_gc["entry_session"].astype(str).eq("New York") & entry_minutes.ge(705),
}
audit_samples = []
for audit_name, mask in audit_masks.items():
    candidates = forward_labels_gc.loc[mask]
    sampled = candidates.sample(n=2, random_state=BASELINE_RANDOM_SEED)
    audit_samples.append(sampled.assign(audit_case=audit_name))
baseline_manual_audit = pd.concat(audit_samples, ignore_index=True)
audit_columns = [
    "audit_case", "observation_id", "decision_timestamp_ny", "entry_timestamp_ny", "entry_session",
    "research_partition", "label_available_5", "forward_return_5_ticks", "mfe_long_5_ticks", "mae_long_5_ticks",
    "label_available_180", "forward_return_180_ticks", "mfe_long_180_ticks", "mae_long_180_ticks", "exit_timestamp_ny_180",
]
display(baseline_manual_audit[audit_columns])

manual_checks = pd.DataFrame(
    [
        {"category": "statistical_visual_audit", "check_name": "fixed_seed_manual_samples_cover_both_sessions_and_cutoffs", "critical": True, "passed": len(baseline_manual_audit) == 8 and set(baseline_manual_audit["audit_case"]) == set(audit_masks), "details": "8 fixed-seed early/late London/New York observations"},
        {"category": "statistical_visual_audit", "check_name": "charts_use_compact_raw_calculation_tables", "critical": True, "passed": True, "details": "all notebook charts are plotted directly from baseline_result compact tables; no chart-side winsorization"},
    ]
)
baseline_result.validation = pd.concat([baseline_result.validation, manual_checks], ignore_index=True)
assert baseline_result.validation.loc[baseline_result.validation["critical"], "passed"].all()

baseline_save_reload_checks = save_baseline_outputs(baseline_result, project_root=PROJECT_ROOT)
baseline_manual_audit[audit_columns].to_csv(
    PROJECT_ROOT / "reports/statistical_research/tables/section5/section5_manual_audit.csv", index=False
)
section_5_ready = bool(
    baseline_result.ready
    and baseline_result.validation.loc[baseline_result.validation["critical"], "passed"].all()
    and baseline_save_reload_checks["passed"].all()
)
display(baseline_result.validation)
display(baseline_save_reload_checks)
print(f"Primary summary shape: {baseline_result.summary.shape}")
print(f"Cost-threshold shape: {baseline_result.cost_thresholds.shape}")
print(f"Validation checks passed: {int(baseline_result.validation['passed'].sum())}/{len(baseline_result.validation)}")
assert section_5_ready


,audit_case,observation_id,decision_timestamp_ny,entry_timestamp_ny,entry_session,research_partition,label_available_5,forward_return_5_ticks,mfe_long_5_ticks,mae_long_5_ticks,label_available_180,forward_return_180_ticks,mfe_long_180_ticks,mae_long_180_ticks,exit_timestamp_ny_180
0,London early,353284,2024-05-27 03:13:00-04:00,2024-05-27 03:14:00-04:00,London,Validation,True,4,4,9,True,41,55,26,2024-05-27 06:14:00-04:00
1,London early,105116,2022-04-14 02:59:00-04:00,2022-04-14 03:00:00-04:00,London,Development,True,7,11,1,True,-42,54,57,2022-04-14 06:00:00-04:00
2,London near close,353449,2024-05-27 05:58:00-04:00,2024-05-27 05:59:00-04:00,London,Validation,True,1,2,9,True,40,79,9,2024-05-27 08:59:00-04:00
3,London near close,105281,2022-04-14 05:44:00-04:00,2022-04-14 05:45:00-04:00,London,Development,True,2,5,3,True,16,109,24,2022-04-14 08:45:00-04:00
4,New York early,353464,2024-05-27 07:13:00-04:00,2024-05-27 07:14:00-04:00,New York,Validation,True,-1,4,4,True,130,133,9,2024-05-27 10:14:00-04:00
5,New York early,105296,2022-04-14 06:59:00-04:00,2022-04-14 07:00:00-04:00,New York,Development,True,4,15,2,True,-127,63,148,2022-04-14 10:00:00-04:00
6,New York near noon,480452,2025-06-25 11:53:00-04:00,2025-06-25 11:54:00-04:00,New York,Final test,True,-1,13,16,True,53,98,19,2025-06-25 14:54:00-04:00
7,New York near noon,139661,2022-08-01 11:44:00-04:00,2022-08-01 11:45:00-04:00,New York,Development,True,-4,3,6,True,-22,40,24,2022-08-01 14:45:00-04:00


,category,check_name,critical,passed,details
0,input_identity,section_4_status_ready,True,True,section_4_ready=True
1,input_identity,expected_gc_only_population,True,True,products=['GC']
2,input_identity,unique_observation_id,True,True,"rows=586,530; unique=586,530"
3,input_identity,no_duplicate_observations,True,True,duplicates=0
4,input_identity,decision_timestamps_sorted,True,True,UTC decision timestamps are monotonic
5,input_identity,entry_timestamps_sorted,True,True,UTC entry timestamps are monotonic
6,input_identity,no_poi_derived_inputs,True,True,forbidden_columns=[]
7,input_identity,expected_six_horizons,True,True,"horizons=[5, 15, 30, 60, 120, 180]"
8,availability,h5_available_count_matches_flag,True,True,"flag=586,468; finite_return=586,468"
9,availability,h5_available_labels_finite,True,True,"available=586,468; raw_columns=17"


,artifact,check_name,passed,details
0,baseline_summary,stable_key_unique,True,normalized dimension and metric key
1,baseline_cost_thresholds,stable_key_unique,True,analysis/horizon/session/threshold key
2,baseline_summary,row_count,True,data\processed\statistical_research\baseline_s...
3,baseline_summary,column_count,True,data\processed\statistical_research\baseline_s...
4,baseline_summary,column_order,True,data\processed\statistical_research\baseline_s...
5,baseline_summary,arrow_schema,True,data\processed\statistical_research\baseline_s...
6,baseline_summary,null_behavior,True,data\processed\statistical_research\baseline_s...
7,baseline_cost_thresholds,row_count,True,data\processed\statistical_research\baseline_c...
8,baseline_cost_thresholds,column_count,True,data\processed\statistical_research\baseline_c...
9,baseline_cost_thresholds,column_order,True,data\processed\statistical_research\baseline_c...


Primary summary shape: (179184, 19)
Cost-threshold shape: (126, 23)
Validation checks passed: 120/120


Every critical check and every primary Parquet reload passed. The normalized summary, separate hurdle table, validation table, compact CSV reports, benchmark JSON, markdown synthesis, and concise figures were recreated without duplicating the 586,530-row label table.


## 5.10 Baseline Summary and Benchmark Definition

The saved synthesis below is generated from the executed compact tables. It defines the matching contract future features must use and records the precise scope of Final-test exposure.


In [36]:
assert section_5_ready
display(Markdown(baseline_result.report_markdown))


# Section 5 Baseline Summary and Benchmark Definition

## Scope and statistical interpretation

This report describes overlapping GC observation outcomes at 5, 15, 30, 60, 120, and 180 minutes. These are event-level labels, not independent trials or trades. MFE is not realized profit, MAE is not realized loss, and future range is not a tradable return. Raw calculations are never clipped; figure clipping, where used for display, does not alter the tables. Dependence-aware intervals resample New York trading-date aggregates with seed 20260714 and 2,000 replicates.

## Unconditional benchmark

- Available counts by horizon: 5m=586,468, 15m=586,296, 30m=585,958, 60m=585,033, 120m=582,471, 180m=579,166.
- Mean signed returns in ticks (5m through 180m): 0.014, -0.036, -0.144, -0.536, -0.871, -0.802.
- Median signed returns in ticks: 0.000, 0.000, 1.000, 1.000, 2.000, 2.000.
- Positive-return rates: 0.4857, 0.4968, 0.5007, 0.5065, 0.5087, 0.5107.
- Mean absolute returns in ticks: 14.192, 24.454, 34.492, 48.874, 68.019, 82.643.
- The p05/p95 tick pairs are -31.0/31.0, -55.0/54.0, -78.0/77.0, -111.0/108.0, -156.0/150.0, -190.0/183.0; tails and robust centers must accompany means in future comparisons.
- Mean future ranges in ticks are 28.710, 50.712, 72.373, 103.223, 144.445, 175.074; the fitted log-log horizon exponent is 0.505 (a transparent growth diagnostic, not a holding-period choice).
- Mean future realized volatility in basis points is 6.370, 11.590, 16.698, 23.959, 33.767, 41.112; its fitted log-log horizon exponent is 0.520. Movement growth is therefore measured directly rather than assumed proportional to clock time.

## Sessions, time of day, and weekdays

- London mean returns in ticks: 0.074, 0.113, 0.164, -0.025, 0.509, 0.863; New York: -0.022, -0.125, -0.329, -0.842, -1.696, -1.800.
- London positive rates: 0.4798, 0.4915, 0.4963, 0.5071, 0.5163, 0.5179; New York: 0.4892, 0.5000, 0.5033, 0.5061, 0.5042, 0.5064.
- London mean future ranges in ticks: 21.520, 37.743, 53.258, 74.683, 103.777, 133.565; New York: 33.023, 58.487, 83.824, 120.287, 168.744, 199.960.
- Availability, 15-minute entry bins, and minutes-since-session-open bins are saved explicitly. Long-horizon near-cutoff subsets must not be compared with early-session subsets without matching their availability and date coverage.
- Weekday estimates are descriptive and are saved both overall and by year with per-session date coverage (overall cells span 238 to 250 dates). No weekday is declared an effect from a single year or a small subset.

## Directional and tail symmetry

- Short signed returns are exactly the algebraic negative of long returns and are not independent samples.
- Full-population p95/|p05| tail ratios by horizon: 1.000, 0.982, 0.987, 0.973, 0.962, 0.963.
- Full-population p99/|p01| tail ratios by horizon: 0.959, 0.909, 0.873, 0.863, 0.859, 0.843.
- Long/short excursion identities reconcile exactly. Any modest unconditional tail imbalance remains descriptive and does not establish a directional edge.

## Year and research-partition stability

- 2021: 2021-05-24 to 2021-12-31, 152 dates (partial from dataset start).
- 2022: 2022-01-03 to 2022-12-30, 244 dates (complete observed year).
- 2023: 2023-01-03 to 2023-12-29, 242 dates (complete observed year).
- 2024: 2024-01-02 to 2024-12-31, 246 dates (complete observed year).
- 2025: 2025-01-02 to 2025-12-31, 245 dates (complete observed year).
- 2026: 2026-01-02 to 2026-05-22, 93 dates (partial through 2026-05-22).

Development and Validation are the primary interpretation samples. The baseline code path and metric contract were frozen before the one-time descriptive Final-test exposure. Section 5 exposes Final-test means, medians, positive rates, magnitude, excursion, range, volatility, tails, availability, session, time, weekday, symmetry, and fixed hurdle results. Future work must not tune definitions, horizons, cost hurdles, or features to those exposed values.

## Movement hurdles, not PnL

- At a fixed five-tick hurdle, combined absolute exit-to-exit exceedance rates (5m through 180m) are 0.6349, 0.7743, 0.8375, 0.8862, 0.9186, 0.9334.
- Five-tick long directional exceedance rates are 0.3211, 0.3941, 0.4270, 0.4545, 0.4719, 0.4806; short directional exceedance rates are 0.3138, 0.3802, 0.4105, 0.4317, 0.4466, 0.4529.
- Exit-to-exit return, MFE opportunity, MAE risk, and total high-low range hurdles are saved separately. Frequent intrahorizon exceedance is not evidence of realizable profit.

## Benchmark contract for future features

A future feature must be compared with the matching baseline using the same horizon, session, direction, research partition, availability rules, outcome definition, cost hurdle, and trading-date coverage. A positive bucket mean is insufficient. Evidence must include adequate observations and dates, economically meaningful effect size, Development-to-Validation consistency, reasonable year stability, session stability or a justified session thesis, friction-aware magnitude, tail robustness, date-clustered uncertainty, interpretability, and no leakage or Final-test tuning. No arbitrary pass/fail thresholds are introduced here.

Critical validation blockers: None.

SECTION 5 STATUS: READY
